# Damping from the Escape Probability

Reproduction notebook for *"Damping from the escape probability: a free
calibration rule for diffusion-based reachability ranking in uncertain
graphs"* — I. Konstantellos, CEID, University of Patras.

---

## The problem

Rank vertices of a probabilistic graph by expected reachable mass

$$y^*_i=\mathbb{E}\Big[\sum_{v\in C(i)}x_v\Big],$$

which is exactly single-seed node-weighted influence spread under the
Independent Cascade model, and exactly Edge Percolated Component centrality in
the uniform case. Computing it is #P-hard, so the field uses fast heuristics —
and those heuristics do not agree on a winner.

## The result

The **mean escape probability** $\bar P_{esc}$ — the average probability a
vertex has at least one incident edge present — is computed while the transition
operator is built, so it is free. It predicts the best damping factor for
PageRank-style diffusions:

$$\alpha=\mathrm{clip}(0.502+0.558\,\bar P_{esc},\,0.4,\,1)$$

fitted at $r=+0.965$, recovering $>98\%$ of a per-instance oracle and beating
every constant $\alpha$ — including the best one selectable with hindsight
($0.894$ vs $0.677$). It refits to a second operator family ($r=+0.961$),
survives a $14\times$ change of scale, and still beats every constant on real
networks.

## Two secondary results, both bounding the first

- **$P_{esc}$ is not a new score.** It is exactly the correlation-free
  ($\eta=1$) zeroth iterate of Karrer–Newman–Zdeborová message-passing
  percolation (§14). Iterating beats it. What is worth recording is that the
  *ranking* signal saturates after two sweeps.
- **Routing among heuristics by $\bar P_{esc}$ fails on real data** (§9, §13).
  Works on synthetic graphs, collapses on citation networks. Reported as a
  negative result.
- **The modern RR-set estimator trails plain Monte Carlo for whole-graph
  ranking** at matched budget (§15) — one sampled world informs every vertex,
  one RR set contributes a single containment bit per vertex. It is the right
  primitive for top-$k$ seed selection, not for this objective.

## Contents

| § | content | runtime |
|---|---|---|
| 1 | library | — |
| 2 | the two identities, verified numerically | ~2 min |
| 3 | instances and ground truth | <1 min |
| 4 | damping calibration at $n=64$ | ~3 min |
| 5 | held-out evaluation, full baseline set | ~4 min |
| 6 | does the $\alpha$ rule transfer across operators? | ~6 min |
| 7 | cost, scale, and the Monte-Carlo budget | ~4 min |
| 8 | the stratified 400-instance panel at $n=900$ | ~8 min |
| 9 | **the router** | ~1 min |
| 10 | diagnostics: three claims put to the test | ~10 min |
| 11 | per-node damping: a ceiling, and a negative result | ~15 min |
| 12 | figures | <1 min |
| 13 | real-world validation | ~10 min |
| 14 | message passing: the proper deterministic competitor | ~5 min |
| 15 | the reverse-reachable-set baseline, all three panels | ~15 min |

End to end: roughly one hour on one core. Sections are independent once §1–§5
have run; §8 is checkpointed and resumable.

## 1. The library

Two fairness decisions are baked in and matter for the conclusions.

* **IRIE** gets a spectral clamp $\alpha_{\text{eff}}=\min(\alpha,0.95/\rho(P))$.
  Its default $\alpha=0.7$ assumes weighted-cascade probabilities with row sums
  near one; ours come from a fixed range, so without the clamp the Neumann
  series diverges and the baseline would be sabotaged rather than tested.
* **Degree discount** is implemented in full rather than asserted to reduce to
  degree for a single seed, so the reduction can be *checked*.

In [ ]:
%%writefile escape_lib.py
"""
escape_lib.py
Core library for "Escape-Scaled Diffusion for Reachability Ranking in
Probabilistic Graphs with Uncertain Targets".

Contains:
  * instance generation (ER / BA / grid / Watts-Strogatz potential supports,
    with independent edge-presence probabilities drawn from a range)
  * Monte-Carlo ground truth for y*_i = E[ sum_{v in C(i)} x_v ]
  * every ranking method under comparison, including the influence-maximisation
    proxies (IRIE, MIA, degree-discount) and the sampling estimators
    (RIS / plain Monte Carlo) that the first version of this note omitted.

All scoring functions share the signature  f(inst, **kw) -> np.ndarray of scores.
"""

import time
import numpy as np
import networkx as nx
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr

# --------------------------------------------------------------------------
# Instances
# --------------------------------------------------------------------------


class Instance:
    """An uncertain graph with an uncertain-target prior.

    Attributes
    ----------
    n     : number of vertices
    ei,ej : endpoint arrays of the m undirected potential edges (ei < ej)
    p     : per-edge presence probability, shape (m,)
    x     : target prior, shape (n,), sums to 1
    P     : (n,n) sparse symmetric matrix of presence probabilities
    """

    def __init__(self, n, ei, ej, p, x, meta=None):
        self.n = n
        self.ei = np.asarray(ei)
        self.ej = np.asarray(ej)
        self.p = np.asarray(p, dtype=float)
        self.x = np.asarray(x, dtype=float)
        self.meta = meta or {}
        self.m = len(self.p)
        # symmetric probability matrix
        rows = np.concatenate([self.ei, self.ej])
        cols = np.concatenate([self.ej, self.ei])
        vals = np.concatenate([self.p, self.p])
        self.P = sp.csr_matrix((vals, (rows, cols)), shape=(n, n))
        self._pesc = None
        self._W_cache = {}

    # ---- cheap statistics -------------------------------------------------
    @property
    def pesc(self):
        """P_esc(i) = 1 - prod_j (1 - p_ij): probability i is not isolated."""
        if self._pesc is None:
            L = sp.csr_matrix(
                (np.log1p(-np.clip(self.P.data, 0, 1 - 1e-15)),
                 self.P.indices, self.P.indptr),
                shape=self.P.shape,
            )
            self._pesc = 1.0 - np.exp(np.asarray(L.sum(axis=1)).ravel())
        return self._pesc

    @property
    def mean_pesc(self):
        return float(self.pesc.mean())

    def W(self, weighting="log"):
        """Row-stochastic transition matrix on the potential support."""
        if weighting in self._W_cache:
            return self._W_cache[weighting]
        d = self.P.data
        if weighting == "log":
            w = -np.log1p(-np.clip(d, 0, 1 - 1e-15))
        elif weighting == "linear":
            w = d.copy()
        elif weighting == "uniform":
            w = np.ones_like(d)
        else:
            raise ValueError(weighting)
        A = sp.csr_matrix((w, self.P.indices, self.P.indptr), shape=self.P.shape)
        rs = np.asarray(A.sum(axis=1)).ravel()
        inv = np.where(rs > 0, 1.0 / np.where(rs > 0, rs, 1.0), 0.0)
        Wm = sp.diags(inv) @ A
        self._W_cache[weighting] = Wm.tocsr()
        return self._W_cache[weighting]


def _prior(n, rng, kind="dirichlet"):
    if kind == "dirichlet":
        x = rng.dirichlet(np.full(n, 0.7))
    elif kind == "uniform":
        x = np.full(n, 1.0 / n)
    elif kind == "sparse":
        x = np.zeros(n)
        k = max(1, n // 10)
        idx = rng.choice(n, size=k, replace=False)
        x[idx] = rng.dirichlet(np.full(k, 1.0))
    else:
        raise ValueError(kind)
    return x / x.sum()


def make_instance(topology, n, p_lo, p_hi, seed, prior_kind="dirichlet",
                  avg_deg=6):
    """Build a potential support of the requested family and attach edge
    probabilities drawn uniformly from [p_lo, p_hi] and a target prior."""
    rng = np.random.default_rng(seed)
    if topology == "er":
        G = nx.gnp_random_graph(n, avg_deg / (n - 1), seed=int(seed))
    elif topology == "ba":
        G = nx.barabasi_albert_graph(n, max(1, avg_deg // 2), seed=int(seed))
    elif topology == "grid":
        side = int(round(np.sqrt(n)))
        assert side * side == n, f"grid needs square n, got {n}"
        G = nx.convert_node_labels_to_integers(nx.grid_2d_graph(side, side))
    elif topology == "ws":
        G = nx.watts_strogatz_graph(n, avg_deg, 0.1, seed=int(seed))
    else:
        raise ValueError(topology)
    ei, ej = [], []
    for u, v in G.edges():
        a, b = (u, v) if u < v else (v, u)
        ei.append(a)
        ej.append(b)
    ei = np.array(ei, dtype=np.int32)
    ej = np.array(ej, dtype=np.int32)
    p = rng.uniform(p_lo, p_hi, size=len(ei))
    x = _prior(n, rng, prior_kind)
    return Instance(n, ei, ej, p, x,
                    meta=dict(topology=topology, n=n, p_lo=p_lo, p_hi=p_hi,
                              seed=seed, prior=prior_kind))


# --------------------------------------------------------------------------
# Ground truth  (== single-seed weighted influence spread under IC,
#                == weighted Edge Percolated Component score)
# --------------------------------------------------------------------------


def _sample_component_mass(inst, rng):
    """One possible world; return the component mass vector."""
    keep = rng.random(inst.m) < inst.p
    ei, ej = inst.ei[keep], inst.ej[keep]
    A = sp.coo_matrix((np.ones(len(ei)), (ei, ej)), shape=(inst.n, inst.n))
    ncomp, lab = connected_components(A, directed=False)
    mass = np.bincount(lab, weights=inst.x, minlength=ncomp)
    return mass[lab]


def ground_truth(inst, n_samples=3000, seed=0):
    """Monte-Carlo estimate of y*_i = E[ sum_{v in C(i)} x_v ]."""
    rng = np.random.default_rng(seed)
    acc = np.zeros(inst.n)
    for _ in range(n_samples):
        acc += _sample_component_mass(inst, rng)
    return acc / n_samples


# --------------------------------------------------------------------------
# Ranking methods
# --------------------------------------------------------------------------


def _normalise(v):
    s = v.sum()
    return v / s if s > 0 else v


def diffuse(inst, alpha, use_pesc=True, orientation="pull", weighting="log",
            tol=1e-13, max_iter=800):
    """The iteration of the paper.

        pull:  y <- normalise[(1-a) x + a (Pesc . (W y))]
        push:  y <- normalise[(1-a) x + a (Pesc . (W^T y))]

    With use_pesc=False and orientation='push' this is standard personalised
    PageRank; with use_pesc=False and orientation='pull' it is its adjoint.
    """
    W = inst.W(weighting)
    Op = W if orientation == "pull" else W.T.tocsr()
    pe = inst.pesc if use_pesc else 1.0
    x = inst.x
    y = x.copy()
    for _ in range(max_iter):
        z = Op @ y
        if use_pesc:
            z = pe * z
        y_new = _normalise((1.0 - alpha) * x + alpha * z)
        if np.abs(y_new - y).max() < tol:
            y = y_new
            break
        y = y_new
    return y


ALPHA_RULE = (0.544, 0.494)


def alpha_from_rule(inst, coef=ALPHA_RULE):
    a, b = coef
    return float(np.clip(a + b * inst.mean_pesc, 0.4, 1.0))


# ---- trivial / structural baselines ---------------------------------------

def m_prior(inst):
    return inst.x.copy()


def m_sum_p(inst):
    return np.asarray(inst.P.sum(axis=1)).ravel()


def m_pesc(inst):
    return inst.pesc.copy()


def m_degree(inst):
    """Plain potential degree."""
    return np.asarray((inst.P > 0).sum(axis=1)).ravel().astype(float)


def m_degree_discount(inst):
    """Chen, Wang & Yang (KDD 2009), DegreeDiscountIC, evaluated for a
    single seed.  The discount term t_v counts already-selected neighbours;
    for |S| = 1 it is identically zero, so the score reduces exactly to the
    (weighted) degree.  Implemented explicitly so the reduction can be
    verified numerically rather than merely asserted."""
    t = np.zeros(inst.n)                       # no seed selected yet
    d = np.asarray(inst.P.sum(axis=1)).ravel()  # expected degree
    pbar = np.asarray(inst.P.sum(axis=1)).ravel() / np.maximum(
        1, np.asarray((inst.P > 0).sum(axis=1)).ravel())
    return d - 2 * t - (d - t) * t * pbar


# ---- PageRank family ------------------------------------------------------

def m_ppr_pull(inst, alpha=0.85, weighting="log"):
    return diffuse(inst, alpha, use_pesc=False, orientation="pull",
                   weighting=weighting)


def m_ppr_push(inst, alpha=0.85, weighting="log"):
    return diffuse(inst, alpha, use_pesc=False, orientation="push",
                   weighting=weighting)


def m_pull_times_pesc(inst, alpha=None, weighting="log"):
    a = alpha_from_rule(inst) if alpha is None else alpha
    return diffuse(inst, a, use_pesc=False, orientation="pull",
                   weighting=weighting) * inst.pesc


def m_escape_pull(inst, alpha=None, weighting="log"):
    a = alpha_from_rule(inst) if alpha is None else alpha
    return diffuse(inst, a, use_pesc=True, orientation="pull",
                   weighting=weighting)


# ---- influence-maximisation proxies (the missing baselines) ---------------

def _spectral_radius(M, iters=60, seed=0):
    rng = np.random.default_rng(seed)
    v = rng.random(M.shape[0])
    v /= np.linalg.norm(v)
    lam = 0.0
    for _ in range(iters):
        w = M @ v
        nw = np.linalg.norm(w)
        if nw == 0:
            return 0.0
        v = w / nw
        lam = nw
    return float(lam)


def m_irie(inst, alpha=0.7, n_iter=400, tol=1e-12, safety=0.95):
    """IRIE influence rank (Jung, Heo & Chen, ICDM 2012), node-weighted.

    Original:      r(u) = 1 + a * sum_v p_uv r(v)
    Weighted here: r    = x + a * P r        (x is the target prior)

    P is the raw probability matrix, whose row sums may exceed one; that
    departure from stochasticity is exactly IRIE's design point.  It also
    means the Neumann series diverges when a * rho(P) >= 1.  IRIE's default
    a = 0.7 assumes weighted-cascade probabilities (p ~ 1/deg, row sums ~ 1);
    our p_ij are drawn from a fixed range, so we clamp
        a_eff = min(a, safety / rho(P))
    which is the natural spectral analogue of the original setting and keeps
    the comparison fair rather than letting the baseline blow up.
    """
    rp = _spectral_radius(inst.P)
    a_eff = min(alpha, safety / rp) if rp > 0 else alpha
    x = inst.x
    r = x.copy()
    for _ in range(n_iter):
        r_new = x + a_eff * (inst.P @ r)
        if np.abs(r_new - r).max() < tol * max(1.0, np.abs(r).max()):
            r = r_new
            break
        r = r_new
    return r


def m_irie_wc(inst, alpha=0.7, n_iter=400, tol=1e-12, weighting="linear"):
    """IRIE on a row-normalised support -- the weighted-cascade regime the
    method was designed for (row sums exactly 1, so any alpha < 1 converges).
    """
    W = inst.W(weighting)
    x = inst.x
    r = x.copy()
    for _ in range(n_iter):
        r_new = x + alpha * (W @ r)
        if np.abs(r_new - r).max() < tol * max(1.0, np.abs(r).max()):
            r = r_new
            break
        r = r_new
    return r


def m_mia(inst, theta=1e-3):
    """MIA / PMIA-style score (Chen, Wang & Wang, KDD 2010), node-weighted.

    sigma(u) ~= sum_v x_v * pp(u,v) where pp(u,v) is the maximum-probability
    path, truncated at pp > theta.  Computed as a Dijkstra on -log p.
    """
    d = np.clip(inst.P.data, 1e-15, 1.0)
    Wd = sp.csr_matrix((-np.log(d), inst.P.indices, inst.P.indptr),
                       shape=inst.P.shape)
    from scipy.sparse.csgraph import dijkstra
    lim = -np.log(theta)
    D = dijkstra(Wd, directed=False, limit=lim)
    PP = np.exp(-D)
    PP[~np.isfinite(D)] = 0.0
    np.fill_diagonal(PP, 1.0)
    return PP @ inst.x


# ---- sampling estimators (the honest topline) -----------------------------

def m_mc(inst, n_samples=50, seed=12345):
    """Plain Monte-Carlo estimate of y* with a small sample budget.
    This is the same estimator as the ground truth, just cheaper."""
    rng = np.random.default_rng(seed)
    acc = np.zeros(inst.n)
    for _ in range(n_samples):
        acc += _sample_component_mass(inst, rng)
    return acc / n_samples


def m_ris(inst, n_samples=50, seed=999):
    """Reverse-reachable-set estimator (Borgs et al., SODA 2014) specialised
    to this objective.  Sample v ~ x, sample a world, take RR(v) = C(v);
    then P(u in RR) = sum_v x_v P(u ~ v) = y*_u exactly."""
    rng = np.random.default_rng(seed)
    hits = np.zeros(inst.n)
    for _ in range(n_samples):
        v = rng.choice(inst.n, p=inst.x)
        keep = rng.random(inst.m) < inst.p
        ei, ej = inst.ei[keep], inst.ej[keep]
        A = sp.coo_matrix((np.ones(len(ei)), (ei, ej)), shape=(inst.n, inst.n))
        _, lab = connected_components(A, directed=False)
        hits += (lab == lab[v])
    return hits / n_samples


# --------------------------------------------------------------------------
# Evaluation helpers
# --------------------------------------------------------------------------


def rho(score, truth):
    r = spearmanr(score, truth).statistic
    return 0.0 if not np.isfinite(r) else float(r)


def timed(fn, inst, repeats=1, **kw):
    t0 = time.perf_counter()
    for _ in range(repeats):
        s = fn(inst, **kw)
    dt = (time.perf_counter() - t0) / repeats
    return s, dt


def oracle_alpha(inst, truth, grid=None, weighting="log"):
    grid = grid if grid is not None else np.round(np.arange(0.40, 1.001, 0.05), 3)
    best, ba = -2.0, 1.0
    for a in grid:
        r = rho(m_escape_pull(inst, alpha=float(a), weighting=weighting), truth)
        if r > best:
            best, ba = r, float(a)
    return ba, best


In [ ]:
import json, time, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, pearsonr, spearmanr
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components

from escape_lib import *

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 120, "font.size": 9})
print("ready")

## 2. What we are actually ranking

Naming the objective correctly decides which methods are its competitors.
Both identities are checked numerically rather than asserted.

In [ ]:
def ic_spread(inst, u, n_sim, rng):
    """Independent-Cascade single-seed weighted spread, simulated directly."""
    tot = 0.0
    for _ in range(n_sim):
        keep = rng.random(inst.m) < inst.p
        ei, ej = inst.ei[keep], inst.ej[keep]
        A = sp.coo_matrix((np.ones(len(ei)), (ei, ej)), shape=(inst.n, inst.n))
        _, lab = connected_components(A, directed=False)
        tot += inst.x[lab == lab[u]].sum()
    return tot / n_sim

inst = make_instance("er", 60, 0.10, 0.30, 7)
gt   = ground_truth(inst, 40000, seed=3)
rng  = np.random.default_rng(0)
sel  = np.arange(0, inst.n, 6)
ic   = np.array([ic_spread(inst, u, 4000, rng) for u in sel])
print("(a) y* vs simulated IC single-seed weighted spread")
print(f"    max abs discrepancy : {np.abs(gt[sel]-ic).max():.4f}")
print(f"    Monte-Carlo noise   : {1/np.sqrt(4000):.4f}  <- inside the noise")

inst_u = make_instance("er", 60, 0.25, 0.25, 7, prior_kind="uniform")
gt_u   = ground_truth(inst_u, 40000, seed=4)
print("\n(b) EPC identity, uniform prior and constant p")
print(f"    Spearman(n*y*, y*) = {rho(gt_u*inst_u.n, gt_u):.6f}")

**Consequence.** The competitor set is the influence-estimation literature —
degree discount, MIA/PMIA, IRIE, and the sampling estimators — not the PageRank
literature alone.

## 3. Instances and ground truth

Four potential-support families (Erdős–Rényi, Barabási–Albert, square grid,
Watts–Strogatz); each potential edge carries $p_{ij}\sim U[p_{lo},p_{hi}]$;
the target prior is Dirichlet(0.7) unless stated otherwise.

In [ ]:
ins = make_instance("ws", 100, 0.18, 0.38, 101)
g   = ground_truth(ins, 3000, seed=5101)
print(f"n={ins.n}  m={ins.m}  mean P_esc={ins.mean_pesc:.3f}  "
      f"alpha_rule={alpha_from_rule(ins):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(8.5, 3))
ax[0].scatter(ins.pesc, g, s=14, alpha=.7); ax[0].set_xlabel("$P_{esc}$")
ax[0].set_ylabel("$y^*$"); ax[0].set_title("escape probability vs ground truth")
ax[1].scatter(ins.x, g, s=14, alpha=.7, color="tab:red")
ax[1].set_xlabel("prior $x$"); ax[1].set_title("prior vs ground truth")
plt.tight_layout(); plt.show()

## 4. The damping calibration

54 training instances at $n=64$. For each, grid-search the $\alpha$ maximising
Spearman $\rho$ against ground truth, then regress it on $\bar P_{esc}$.

In [ ]:
import json
import numpy as np
from scipy.stats import pearsonr, spearmanr
from escape_lib import make_instance, ground_truth, oracle_alpha

CAL_TOPOS = ["er", "ba", "grid"]
CAL_RANGES = [(0.01, 0.05), (0.03, 0.10), (0.07, 0.20),
              (0.15, 0.35), (0.25, 0.50), (0.40, 0.70)]
CAL_SEEDS = [11, 12, 13]
N_CAL = 64
MC_CAL = 2500

rows = []
for topo in CAL_TOPOS:
    for (lo, hi) in CAL_RANGES:
        for sd in CAL_SEEDS:
            inst = make_instance(topo, N_CAL, lo, hi, sd)
            gt = ground_truth(inst, MC_CAL, seed=1000 + sd)
            a_opt, r_opt = oracle_alpha(inst, gt)
            rows.append(dict(topology=topo, lo=lo, hi=hi, seed=sd,
                             mean_pesc=inst.mean_pesc,
                             alpha_opt=a_opt, rho_opt=r_opt))
            print(f"{topo:5s} p[{lo:.2f},{hi:.2f}] s{sd}  "
                  f"Pesc={inst.mean_pesc:.3f}  a*={a_opt:.2f}  rho={r_opt:.3f}",
                  flush=True)

pe = np.array([r["mean_pesc"] for r in rows])
al = np.array([r["alpha_opt"] for r in rows])
b, a = np.polyfit(pe, al, 1)
pred = np.clip(a + b * pe, 0.4, 1.0)
rmse = float(np.sqrt(np.mean((pred - al) ** 2)))
pr = pearsonr(pe, al)
sr = spearmanr(pe, al)

print("\n=== CALIBRATION FIT ===")
print(f"instances       : {len(rows)}")
print(f"alpha = clip({a:.3f} + {b:.3f} * mean_Pesc, 0.4, 1.0)")
print(f"Pearson r       : {pr.statistic:+.3f}  (p={pr.pvalue:.2e})")
print(f"Spearman rho    : {sr.statistic:+.3f}")
print(f"residual RMSE   : {rmse:.3f}")

COEF = (float(a), float(b))
json.dump(dict(rows=rows, intercept=float(a), slope=float(b),
               pearson=float(pr.statistic), pearson_p=float(pr.pvalue),
               spearman=float(sr.statistic), rmse=rmse),
          open("calibration.json", "w"), indent=1)
print("\nwrote calibration.json")

## 5. Held-out evaluation against the full baseline set

100 instances at $n=100$: four topologies including Watts–Strogatz (absent from
calibration) × five probability ranges × five seeds.

In [ ]:
import json
import numpy as np
from scipy.stats import wilcoxon
from escape_lib import (make_instance, ground_truth, rho, timed, oracle_alpha,
                        alpha_from_rule, m_prior, m_sum_p, m_pesc,
                        m_degree_discount, m_ppr_pull, m_ppr_push,
                        m_pull_times_pesc, m_escape_pull, m_irie, m_irie_wc,
                        m_mia, m_mc, m_ris)


TOPOS = ["er", "ba", "grid", "ws"]
RANGES = [(0.02, 0.08), (0.08, 0.18), (0.18, 0.38), (0.28, 0.52), (0.45, 0.72)]
SEEDS = [101, 102, 103, 104, 105]
N = 100
MC = 3000

METHODS = {
    "x only":                    lambda i: m_prior(i),
    "sum_j p_ij":                lambda i: m_sum_p(i),
    "degree discount (KDD09)":   lambda i: m_degree_discount(i),
    "P_esc alone":               lambda i: m_pesc(i),
    "PPR pull a=.85":            lambda i: m_ppr_pull(i, 0.85),
    "PPR pull rule-a":           lambda i: m_ppr_pull(i, alpha_from_rule(i, COEF)),
    "PPR push a=.85":            lambda i: m_ppr_push(i, 0.85),
    "PPR push rule-a":           lambda i: m_ppr_push(i, alpha_from_rule(i, COEF)),
    "pull x P_esc post hoc":     lambda i: m_pull_times_pesc(i, alpha_from_rule(i, COEF)),
    "escape-scaled a=1":         lambda i: m_escape_pull(i, 1.0),
    "IRIE (ICDM12)":             lambda i: m_irie(i, 0.7),
    "IRIE row-normalised":       lambda i: m_irie_wc(i, 0.7),
    "MIA/PMIA (KDD10)":          lambda i: m_mia(i, 1e-3),
    "MC 10 samples":             lambda i: m_mc(i, 10),
    "MC 50 samples":             lambda i: m_mc(i, 50),
    "MC 200 samples":            lambda i: m_mc(i, 200),
    "RIS 50 samples":            lambda i: m_ris(i, 50),
    "THIS WORK":                 lambda i: m_escape_pull(i, alpha_from_rule(i, COEF)),
}

records = []
for topo in TOPOS:
    for (lo, hi) in RANGES:
        for sd in SEEDS:
            inst = make_instance(topo, N, lo, hi, sd)
            gt = ground_truth(inst, MC, seed=5000 + sd)
            rec = dict(topology=topo, lo=lo, hi=hi, seed=sd,
                       mean_pesc=inst.mean_pesc,
                       alpha_rule=alpha_from_rule(inst, COEF), rho={}, ms={})
            for name, fn in METHODS.items():
                s, dt = timed(fn, inst)
                rec["rho"][name] = rho(s, gt)
                rec["ms"][name] = dt * 1000
            ao, ro = oracle_alpha(inst, gt)
            rec["rho"]["oracle a"] = ro
            rec["ms"]["oracle a"] = float("nan")
            rec["alpha_oracle"] = ao
            records.append(rec)
        print(f"done {topo} p[{lo:.2f},{hi:.2f}]", flush=True)

names = list(METHODS.keys()) + ["oracle a"]
M = {nm: np.array([r["rho"][nm] for r in records]) for nm in names}
T = {nm: np.array([r["ms"][nm] for r in records]) for nm in names}
ours = M["THIS WORK"]

print("\n" + "=" * 78)
print(f"{'method':28s} {'mean rho':>9s} {'gain':>8s} {'wins':>6s} "
      f"{'p':>10s} {'ms':>8s}")
print("=" * 78)
summary = []
for nm in sorted(names, key=lambda k: M[k].mean()):
    v = M[nm]
    if nm == "THIS WORK":
        line = (nm, v.mean(), None, None, None, T[nm].mean())
    else:
        d = ours - v
        nz = d[np.abs(d) > 1e-12]
        p = wilcoxon(nz).pvalue if len(nz) > 0 else 1.0
        line = (nm, v.mean(), v.mean() - ours.mean(), int((d > 0).sum()), p,
                T[nm].mean())
    summary.append(line)
    g = "" if line[2] is None else f"{line[2]:+.3f}"
    w = "" if line[3] is None else f"{line[3]}/100"
    pp = "" if line[4] is None else f"{line[4]:.1e}"
    print(f"{nm:28s} {line[1]:9.3f} {g:>8s} {w:>6s} {pp:>10s} {line[5]:8.3f}")

json.dump(dict(records=records, coef=COEF), open("holdout.json", "w"))
print("\nwrote holdout.json")

## 6. Does the $\alpha$ rule describe the operator, or the problem?

The decisive test. Refit the same functional form, independently and from
scratch, for three operators on the *same* calibration instances.

In [ ]:
import json
import numpy as np
from scipy.stats import pearsonr
from escape_lib import (make_instance, ground_truth, rho, diffuse, m_irie_wc,
                        m_escape_pull, m_ppr_push)

CAL_TOPOS = ["er", "ba", "grid"]
CAL_RANGES = [(0.01, 0.05), (0.03, 0.10), (0.07, 0.20),
              (0.15, 0.35), (0.25, 0.50), (0.40, 0.70)]
CAL_SEEDS = [11, 12, 13]
GRID = np.round(np.arange(0.40, 1.001, 0.05), 3)

FAMILIES = {
    "escape-scaled pull": lambda i, a: diffuse(i, a, use_pesc=True,  orientation="pull"),
    "PPR push":           lambda i, a: diffuse(i, a, use_pesc=False, orientation="push"),
    "IRIE row-norm":      lambda i, a: m_irie_wc(i, alpha=a),
}

# ---------- calibrate each family separately ----------
cal = {k: {"pe": [], "a": []} for k in FAMILIES}
for topo in CAL_TOPOS:
    for (lo, hi) in CAL_RANGES:
        for sd in CAL_SEEDS:
            inst = make_instance(topo, 64, lo, hi, sd)
            gt = ground_truth(inst, 2500, seed=1000 + sd)
            for fam, fn in FAMILIES.items():
                best, ba = -2.0, 1.0
                for a in GRID:
                    r = rho(fn(inst, float(a)), gt)
                    if r > best:
                        best, ba = r, float(a)
                cal[fam]["pe"].append(inst.mean_pesc)
                cal[fam]["a"].append(ba)
    print("cal", topo, flush=True)

rules = {}
print("\n=== per-family alpha rules fitted on the SAME 54 instances ===")
for fam in FAMILIES:
    pe = np.array(cal[fam]["pe"])
    al = np.array(cal[fam]["a"])
    b, a = np.polyfit(pe, al, 1)
    r = pearsonr(pe, al)
    rules[fam] = (float(a), float(b))
    rm = float(np.sqrt(np.mean((np.clip(a + b * pe, .4, 1) - al) ** 2)))
    print(f"{fam:22s} alpha=clip({a:.3f}+{b:.3f}*Pesc,.4,1)  "
          f"r={r.statistic:+.3f}  RMSE={rm:.3f}")

# ---------- test out of sample ----------
TOPOS = ["er", "ba", "grid", "ws"]
RANGES = [(0.02, 0.08), (0.08, 0.18), (0.18, 0.38), (0.28, 0.52), (0.45, 0.72)]
SEEDS = [101, 102, 103, 104, 105]

res = {f"{fam}|{mode}": [] for fam in FAMILIES
       for mode in ("fixed .85", "rule", "oracle")}
for topo in TOPOS:
    for (lo, hi) in RANGES:
        for sd in SEEDS:
            inst = make_instance(topo, 100, lo, hi, sd)
            gt = ground_truth(inst, 3000, seed=5000 + sd)
            for fam, fn in FAMILIES.items():
                a0, b0 = rules[fam]
                ar = float(np.clip(a0 + b0 * inst.mean_pesc, 0.4, 1.0))
                res[f"{fam}|fixed .85"].append(rho(fn(inst, 0.85), gt))
                res[f"{fam}|rule"].append(rho(fn(inst, ar), gt))
                res[f"{fam}|oracle"].append(
                    max(rho(fn(inst, float(a)), gt) for a in GRID))
    print("test", topo, flush=True)

print("\n=== out-of-sample mean Spearman rho, 100 instances ===")
print(f"{'family':22s} {'fixed .85':>10s} {'rule-a':>10s} {'oracle':>10s} "
      f"{'gain':>8s} {'% oracle':>9s}")
out = {}
for fam in FAMILIES:
    f8 = np.mean(res[f"{fam}|fixed .85"])
    fr = np.mean(res[f"{fam}|rule"])
    fo = np.mean(res[f"{fam}|oracle"])
    out[fam] = dict(fixed=f8, rule=fr, oracle=fo,
                    intercept=rules[fam][0], slope=rules[fam][1])
    print(f"{fam:22s} {f8:10.3f} {fr:10.3f} {fo:10.3f} "
          f"{fr - f8:+8.3f} {100 * fr / fo:8.1f}%")

json.dump(dict(rules=rules, results={k: list(map(float, v))
                                     for k, v in res.items()}, summary=out),
          open("alpha_transfer.json", "w"), indent=1)
print("\nwrote alpha_transfer.json")

The rule transfers to both teleportation-style diffusions ($r>0.96$,
$>98\%$ of oracle) and weakly to IRIE's row-normalised variant ($r=+0.767$) —
the limit of its reach, reported rather than omitted.

## 7. Cost, scale, and what the heuristic is worth in Monte-Carlo samples

Both the diffusion and the sampling estimator are $O(\text{iters}\cdot m)$, and
Monte-Carlo error decays as $O(1/\sqrt{W})$ independently of $n$ — so any
advantage is a constant factor, not a complexity class.

In [ ]:
import json
import time
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from escape_lib import (make_instance, ground_truth, rho, timed,
                        alpha_from_rule, diffuse, m_pesc, m_sum_p, m_irie,
                        m_mia, m_mc, m_escape_pull, m_ppr_push)

CAL = json.load(open("calibration.json"))
COEF = (CAL["intercept"], CAL["slope"])
AT = json.load(open("alpha_transfer.json"))
PUSH_COEF = (AT["rules"]["PPR push"][0], AT["rules"]["PPR push"][1])

# ---------------------------------------------------------------- identity
print("=== 1. identity check ===")
inst = make_instance("er", 60, 0.10, 0.30, 7)

# (a) live-edge / IC simulation: activate seed u, spread along live edges
def ic_spread(inst, u, n_sim, rng):
    tot = 0.0
    for _ in range(n_sim):
        keep = rng.random(inst.m) < inst.p
        ei, ej = inst.ei[keep], inst.ej[keep]
        A = sp.coo_matrix((np.ones(len(ei)), (ei, ej)), shape=(inst.n, inst.n))
        _, lab = connected_components(A, directed=False)
        tot += inst.x[lab == lab[u]].sum()
    return tot / n_sim

rng = np.random.default_rng(0)
gt = ground_truth(inst, 40000, seed=3)
ic = np.array([ic_spread(inst, u, 4000, rng) for u in range(0, inst.n, 6)])
sel = np.arange(0, inst.n, 6)
print(f"  max |y* - IC single-seed weighted spread| = "
      f"{np.abs(gt[sel] - ic).max():.4f}  (MC noise ~ 1/sqrt(4000) = "
      f"{1/np.sqrt(4000):.4f})")

# (b) EPC: uniform prior, uniform p -> y* proportional to EPC score
inst_u = make_instance("er", 60, 0.25, 0.25, 7, prior_kind="uniform")
gt_u = ground_truth(inst_u, 40000, seed=4)
epc = gt_u * inst_u.n          # EPC counts nodes, y* averages unit mass
print(f"  EPC(v) = n * y*_v with uniform x and constant p: "
      f"mean EPC = {epc.mean():.3f}, matches by construction "
      f"(Spearman {rho(epc, gt_u):.6f})")

# ---------------------------------------------------------------- scale
print("\n=== 2. cost/accuracy at n=900 ===")
SC_TOPOS = ["er", "ba", "grid", "ws"]
SC_RANGES = [(0.02, 0.08), (0.08, 0.18), (0.28, 0.52)]
SC_SEEDS = [201, 202]
METHODS = {
    "P_esc alone":     lambda i: m_pesc(i),
    "sum_j p_ij":      lambda i: m_sum_p(i),
    "MIA/PMIA":        lambda i: m_mia(i, 1e-3),
    "IRIE":            lambda i: m_irie(i, 0.7),
    "PPR push rule-a": lambda i: diffuse(i, float(np.clip(
        PUSH_COEF[0] + PUSH_COEF[1] * i.mean_pesc, .4, 1)),
        use_pesc=False, orientation="push"),
    "escape rule-a":   lambda i: m_escape_pull(i, alpha_from_rule(i, COEF)),
    "MC 50":           lambda i: m_mc(i, 50),
    "MC 200":          lambda i: m_mc(i, 200),
    "MC 1000":         lambda i: m_mc(i, 1000),
}
acc = {k: [] for k in METHODS}
tim = {k: [] for k in METHODS}
for topo in SC_TOPOS:
    for (lo, hi) in SC_RANGES:
        for sd in SC_SEEDS:
            ins = make_instance(topo, 900, lo, hi, sd)
            g = ground_truth(ins, 4000, seed=7000 + sd)
            for nm, fn in METHODS.items():
                s, dt = timed(fn, ins)
                acc[nm].append(rho(s, g))
                tim[nm].append(dt * 1000)
    print("  scale", topo, flush=True)

print(f"\n  {'method':18s} {'mean rho':>9s} {'ms':>9s}")
scale_summary = {}
for nm in sorted(METHODS, key=lambda k: np.mean(acc[k])):
    scale_summary[nm] = dict(rho=float(np.mean(acc[nm])),
                             ms=float(np.mean(tim[nm])))
    print(f"  {nm:18s} {np.mean(acc[nm]):9.3f} {np.mean(tim[nm]):9.2f}")

# ------------------------------------------------- MC budget to match
print("\n=== 3. Monte-Carlo budget needed to match the heuristic (n=100) ===")
budgets = [5, 10, 20, 40, 80, 160, 320]
rows = []
for topo in ["er", "ba", "grid", "ws"]:
    for (lo, hi) in [(0.02, 0.08), (0.18, 0.38), (0.45, 0.72)]:
        for sd in [301, 302]:
            ins = make_instance(topo, 100, lo, hi, sd)
            g = ground_truth(ins, 3000, seed=9000 + sd)
            r_ours, t_ours = timed(m_escape_pull, ins,
                                   alpha=alpha_from_rule(ins, COEF))
            row = dict(topo=topo, ours=rho(r_ours, g), ours_ms=t_ours * 1000)
            for b in budgets:
                s, dt = timed(m_mc, ins, n_samples=b)
                row[f"mc{b}"] = rho(s, g)
                row[f"mc{b}_ms"] = dt * 1000
            rows.append(row)
print(f"  {'budget':>8s} {'mean rho':>9s} {'mean ms':>9s}")
print(f"  {'ours':>8s} {np.mean([r['ours'] for r in rows]):9.3f} "
      f"{np.mean([r['ours_ms'] for r in rows]):9.3f}")
mc_curve = {}
for b in budgets:
    mc_curve[b] = dict(rho=float(np.mean([r[f"mc{b}"] for r in rows])),
                       ms=float(np.mean([r[f"mc{b}_ms"] for r in rows])))
    print(f"  {b:8d} {mc_curve[b]['rho']:9.3f} {mc_curve[b]['ms']:9.3f}")

json.dump(dict(scale=scale_summary, mc_curve=mc_curve,
               ours_small=dict(rho=float(np.mean([r['ours'] for r in rows])),
                               ms=float(np.mean([r['ours_ms'] for r in rows])))),
          open("scale_identity.json", "w"), indent=1)
print("\nwrote scale_identity.json")

## 8. The stratified panel at $n=900$

A fixed grid samples the density axis only where the grid happens to fall, and
leaves comparisons underpowered. This design stratifies on $\bar P_{esc}$
(eight equal-width strata × 50) and randomises everything else — topology,
average degree, probability range, prior type, seed. Average degree and prior
were *fixed* in the earlier panels; varying them widens the population the
conclusions apply to.

Checkpointed per instance to `followup_n900.jsonl`; re-running skips completed
work.

In [ ]:
%%writefile followup.py
"""
Follow-up study at n=900 -- stratified random design, resumable.

Usage:
    python3 followup.py sample          # draw and save 400 configs
    python3 followup.py run <count>     # evaluate up to <count> pending configs

Results are appended to followup_n900.jsonl, one record per line, so the job
can be resumed after an interruption without losing work.
"""

import json
import os
import sys
import time
import numpy as np
from escape_lib import (make_instance, ground_truth, rho, timed, diffuse,
                        alpha_from_rule, m_pesc, m_sum_p, m_irie, m_mia,
                        m_mc, m_escape_pull)

CFG_FILE = "followup_configs.json"
OUT_FILE = "followup_n900.jsonl"

N = 900
MC_GT = 3000
N_BINS = 8
PER_BIN = 50
TOPOS = ["er", "ba", "grid", "ws"]
DEGS = [4, 6, 8, 10, 12]
PRIORS = ["dirichlet", "uniform", "sparse"]
ALPHA_GRID = np.round(np.arange(0.40, 1.001, 0.025), 4)

CAL = json.load(open("calibration.json"))
COEF = (CAL["intercept"], CAL["slope"])
AT = json.load(open("alpha_transfer.json"))
PUSH_COEF = tuple(AT["rules"]["PPR push"])


def push_rule(inst):
    a = float(np.clip(PUSH_COEF[0] + PUSH_COEF[1] * inst.mean_pesc, 0.4, 1.0))
    return diffuse(inst, a, use_pesc=False, orientation="push")


METHODS = {
    "P_esc alone":     lambda i: m_pesc(i),
    "sum_j p_ij":      lambda i: m_sum_p(i),
    "MIA/PMIA":        lambda i: m_mia(i, 1e-3),
    "IRIE":            lambda i: m_irie(i, 0.7),
    "push PPR rule-a": push_rule,
    "escape rule-a":   lambda i: m_escape_pull(i, alpha_from_rule(i, COEF)),
    "MC 50":           lambda i: m_mc(i, 50),
    "MC 200":          lambda i: m_mc(i, 200),
}


def do_sample():
    """Stratified-random draw: 8 equal-width Pesc strata, 50 instances each.

    Every non-stratifying factor (topology, average degree, probability range,
    prior type, seed) is drawn at random.  Average degree and prior type were
    held FIXED in the original 36-instance grid; varying them here widens the
    population the conclusions apply to.
    """
    rng = np.random.default_rng(20260905)
    bins = [[] for _ in range(N_BINS)]
    attempts = 0
    t0 = time.time()
    while sum(len(b) for b in bins) < N_BINS * PER_BIN and attempts < 60000:
        attempts += 1
        topo = TOPOS[rng.integers(len(TOPOS))]
        deg = int(DEGS[rng.integers(len(DEGS))])
        p_lo = float(np.exp(rng.uniform(np.log(0.005), np.log(0.65))))
        p_hi = float(min(p_lo + rng.uniform(0.02, 0.30), 0.97))
        prior = PRIORS[rng.integers(len(PRIORS))]
        sd = int(rng.integers(1, 10**6))
        inst = make_instance(topo, N, p_lo, p_hi, sd, prior_kind=prior,
                             avg_deg=deg)
        b = min(int(inst.mean_pesc * N_BINS), N_BINS - 1)
        if len(bins[b]) >= PER_BIN:
            continue
        bins[b].append(dict(id=sum(len(z) for z in bins), topology=topo,
                            avg_deg=deg, p_lo=p_lo, p_hi=p_hi, prior=prior,
                            seed=sd, mean_pesc=float(inst.mean_pesc),
                            m=int(inst.m), bin=b))
    cfgs = [c for b in bins for c in b]
    for i, c in enumerate(cfgs):
        c["id"] = i
    json.dump(dict(configs=cfgs, attempts=attempts, n_bins=N_BINS,
                   per_bin=PER_BIN, n=N,
                   alpha_grid=[float(a) for a in ALPHA_GRID],
                   coef_escape=list(COEF), coef_push=list(PUSH_COEF)),
              open(CFG_FILE, "w"))
    print(f"sampled {len(cfgs)} configs in {attempts} attempts "
          f"({time.time()-t0:.1f}s)")
    for k, b in enumerate(bins):
        pes = [c["mean_pesc"] for c in b]
        print(f"  bin {k} Pesc [{k/N_BINS:.3f},{(k+1)/N_BINS:.3f}): "
              f"{len(b):3d}  mean {np.mean(pes):.3f}")


def evaluate(cfg):
    inst = make_instance(cfg["topology"], N, cfg["p_lo"], cfg["p_hi"],
                         cfg["seed"], prior_kind=cfg["prior"],
                         avg_deg=cfg["avg_deg"])
    gt = ground_truth(inst, MC_GT, seed=cfg["seed"] % 10**6 + 1)
    rec = {k: cfg[k] for k in ("id", "topology", "avg_deg", "p_lo", "p_hi",
                               "prior", "seed", "mean_pesc", "m", "bin")}
    rec["rho"] = {}
    rec["ms"] = {}
    for nm, fn in METHODS.items():
        s, dt = timed(fn, inst)
        rec["rho"][nm] = rho(s, gt)
        rec["ms"][nm] = dt * 1000
    ce = np.array([rho(diffuse(inst, float(a), use_pesc=True,
                               orientation="pull"), gt) for a in ALPHA_GRID])
    cp = np.array([rho(diffuse(inst, float(a), use_pesc=False,
                               orientation="push"), gt) for a in ALPHA_GRID])
    rec["oracle_escape"] = float(ce.max())
    rec["alpha_opt_escape"] = float(ALPHA_GRID[int(ce.argmax())])
    rec["oracle_push"] = float(cp.max())
    rec["alpha_opt_push"] = float(ALPHA_GRID[int(cp.argmax())])
    rec["curve_escape"] = [round(float(v), 6) for v in ce]
    rec["curve_push"] = [round(float(v), 6) for v in cp]
    return rec


def do_run(count):
    cfgs = json.load(open(CFG_FILE))["configs"]
    done = set()
    if os.path.exists(OUT_FILE):
        for line in open(OUT_FILE):
            line = line.strip()
            if line:
                done.add(json.loads(line)["id"])
    pending = [c for c in cfgs if c["id"] not in done]
    todo = pending[:count]
    print(f"{len(done)} done, {len(pending)} pending; evaluating {len(todo)}")
    t0 = time.time()
    with open(OUT_FILE, "a") as fh:
        for k, cfg in enumerate(todo):
            rec = evaluate(cfg)
            fh.write(json.dumps(rec) + "\n")
            fh.flush()
            if (k + 1) % 20 == 0:
                el = time.time() - t0
                print(f"  {k+1}/{len(todo)}  {el/60:.1f} min  "
                      f"({el/(k+1):.2f} s/inst)", flush=True)
    print(f"chunk done in {(time.time()-t0)/60:.1f} min; "
          f"total complete {len(done)+len(todo)}/{len(cfgs)}")


if __name__ == "__main__":
    if sys.argv[1] == "sample":
        do_sample()
    elif sys.argv[1] == "run":
        do_run(int(sys.argv[2]))


In [ ]:
subprocess.run([sys.executable, "followup.py", "sample"], check=True)
for _ in range(4):
    subprocess.run([sys.executable, "followup.py", "run", "100"], check=True)

In [ ]:
import json
import numpy as np
from scipy.stats import wilcoxon, pearsonr, spearmanr

CFG = json.load(open("followup_configs.json"))
recs = [json.loads(l) for l in open("followup_n900.jsonl") if l.strip()]
recs.sort(key=lambda r: r["id"])
FU = dict(alpha_grid=CFG["alpha_grid"], coef_escape=CFG["coef_escape"],
          coef_push=CFG["coef_push"], n_bins=CFG["n_bins"],
          per_bin=CFG["per_bin"])
GRID = np.array(FU["alpha_grid"])
OLD = json.load(open("final.json"))          # the original 36-instance panel

METHODS = ["P_esc alone", "sum_j p_ij", "MIA/PMIA", "IRIE",
           "push PPR rule-a", "escape rule-a", "MC 50", "MC 200"]
REF = "escape rule-a"


def boot_ci(d, B=20000, seed=0):
    """Bootstrap 95% CI for the mean of paired differences."""
    r = np.random.default_rng(seed)
    idx = r.integers(0, len(d), size=(B, len(d)))
    means = d[idx].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


R = {m: np.array([x["rho"][m] for x in recs]) for m in METHODS}
T = {m: np.array([x["ms"][m] for x in recs]) for m in METHODS}
pe = np.array([x["mean_pesc"] for x in recs])
ref = R[REF]

print("=" * 96)
print(f"STRATIFIED-RANDOM PANEL, n=900, {len(recs)} instances "
      f"(8 Pesc strata x 50), vs {REF}")
print("=" * 96)
print(f"{'method':18s} {'mean rho':>9s} {'diff':>8s} {'95% CI':>18s} "
      f"{'wins':>9s} {'p':>10s} {'ms':>8s}")
rows = {}
for m in sorted(METHODS, key=lambda k: R[k].mean()):
    v = R[m]
    if m == REF:
        rows[m] = dict(rho=float(v.mean()), ms=float(T[m].mean()))
        print(f"{m:18s} {v.mean():9.4f} {'--':>8s} {'--':>18s} {'--':>9s} "
              f"{'--':>10s} {T[m].mean():8.2f}")
        continue
    d = ref - v
    nz = d[np.abs(d) > 1e-12]
    p = wilcoxon(nz).pvalue if len(nz) else 1.0
    lo, hi = boot_ci(d)
    rows[m] = dict(rho=float(v.mean()), diff=float(v.mean() - ref.mean()),
                   ci=[-hi, -lo], wins=int((d > 0).sum()), p=float(p),
                   ms=float(T[m].mean()))
    print(f"{m:18s} {v.mean():9.4f} {v.mean()-ref.mean():+8.4f} "
          f"[{-hi:+.4f},{-lo:+.4f}]".rjust(0).ljust(0).rjust(19)
          + f" {(d>0).sum():4d}/{len(d)} {p:10.2e} {T[m].mean():8.2f}")

# ---------------------------------------------- resolution of earlier nulls
print("\n" + "=" * 96)
print("DOES THE LARGER SAMPLE RESOLVE THE EARLIER NON-SIGNIFICANT COMPARISONS?")
print("=" * 96)
print(f"{'comparison':28s} {'old p (36)':>12s} {'new p':>12s} {'new diff':>10s} "
      f"{'verdict':>28s}")
oldmap = {"MIA/PMIA": "MIA/PMIA (KDD10)", "IRIE": "IRIE (ICDM12)",
          "push PPR rule-a": "PPR push rule-a", "MC 50": "MC 50",
          "MC 200": "MC 200", "P_esc alone": "P_esc alone",
          "sum_j p_ij": "sum_j p_ij"}
for m in ["MIA/PMIA", "IRIE", "push PPR rule-a", "MC 50", "MC 200"]:
    ok = OLD["n900"].get(oldmap[m], {})
    oldp = ok.get("p", float("nan"))
    newp = rows[m]["p"]
    if oldp > 0.05 and newp < 0.05:
        verd = "null RESOLVED -> real effect"
    elif oldp > 0.05 and newp >= 0.05:
        verd = "still null at 400"
    elif oldp <= 0.05 and newp < 0.05:
        verd = "confirmed"
    else:
        verd = "no longer significant"
    print(f"{REF+' vs '+m:28s} {oldp:12.3f} {newp:12.2e} "
          f"{rows[m]['diff']:+10.4f} {verd:>28s}")

# ---------------------------------------------- stratified breakdown
print("\n" + "=" * 96)
print("BY ESCAPE-PROBABILITY STRATUM (50 instances each)")
print("=" * 96)
show = ["P_esc alone", "MIA/PMIA", "IRIE", "push PPR rule-a",
        "escape rule-a", "MC 200"]
print(f"{'stratum':16s} " + " ".join(f"{s[:14]:>15s}" for s in show) + "   best")
strat = {}
for b in range(FU["n_bins"]):
    sel = np.array([x["bin"] == b for x in recs])
    if sel.sum() == 0:
        continue
    vals = {m: float(R[m][sel].mean()) for m in show}
    best = max(vals, key=vals.get)
    strat[b] = dict(vals=vals, n=int(sel.sum()), best=best)
    lab = f"[{b/FU['n_bins']:.2f},{(b+1)/FU['n_bins']:.2f})"
    print(f"{lab:16s} " + " ".join(f"{vals[m]:15.3f}" for m in show)
          + f"   {best}")

# ---------------------------------------------- does the rule hold at n=900?
print("\n" + "=" * 96)
print("SCALE TRANSFER: the alpha rule was fitted at n=64. Does it hold at n=900?")
print("=" * 96)
for fam, key, coef in [("escape-scaled", "alpha_opt_escape", FU["coef_escape"]),
                       ("push PPR", "alpha_opt_push", FU["coef_push"])]:
    ao = np.array([x[key] for x in recs])
    pr = pearsonr(pe, ao)
    sr = spearmanr(pe, ao)
    b900, a900 = np.polyfit(pe, ao, 1)
    a64, b64 = coef[0], coef[1]
    pred64 = np.clip(a64 + b64 * pe, 0.4, 1.0)
    rmse64 = float(np.sqrt(np.mean((pred64 - ao) ** 2)))
    rmse900 = float(np.sqrt(np.mean((np.clip(a900 + b900 * pe, .4, 1) - ao) ** 2)))
    print(f"\n{fam}:")
    print(f"  n=64  rule      : clip({a64:.3f} + {b64:.3f} Pesc)")
    print(f"  n=900 refit     : clip({a900:.3f} + {b900:.3f} Pesc)")
    print(f"  correlation at n=900 : Pearson {pr.statistic:+.3f} "
          f"(p={pr.pvalue:.1e}), Spearman {sr.statistic:+.3f}")
    print(f"  RMSE of n=64 rule at n=900 : {rmse64:.3f}   "
          f"(refit-at-900 RMSE {rmse900:.3f})")

# ---------------------------------------------- how much does the rule recover
print("\n" + "=" * 96)
print("RULE vs BEST FIXED alpha vs ORACLE, at n=900 (rule fitted at n=64)")
print("=" * 96)
for fam, ck, ok_, rk in [("escape-scaled", "curve_escape", "oracle_escape",
                          "escape rule-a"),
                         ("push PPR", "curve_push", "oracle_push",
                          "push PPR rule-a")]:
    curves = np.array([x[ck] for x in recs])
    mean_curve = curves.mean(axis=0)
    bf, ba = float(mean_curve.max()), float(GRID[int(mean_curve.argmax())])
    orc = float(np.mean([x[ok_] for x in recs]))
    rl = float(R[rk].mean())
    a85 = float(mean_curve[int(np.argmin(np.abs(GRID - 0.85)))])
    print(f"\n{fam}:")
    print(f"  fixed alpha=0.85              : {a85:.4f}")
    print(f"  best fixed alpha (hindsight)  : {bf:.4f}  at alpha={ba:.3f}")
    print(f"  rule-alpha (fitted at n=64)   : {rl:.4f}   "
          f"(+{rl-bf:.4f} over best fixed)")
    print(f"  per-instance oracle           : {orc:.4f}   "
          f"({100*rl/orc:.1f}% of oracle)")

out = dict(n=len(recs), rows=rows, strat=strat)
json.dump(out, open("followup_analysis.json", "w"), indent=1)
print("\nwrote followup_analysis.json")

## 9. The router

No method wins everywhere, and the leader changes systematically with density.
So predict the regime instead of searching for a winner — from a statistic that
costs nothing and is available *before* any algorithm runs.

Evaluation is 5-fold cross-validation, so no bucket assignment is ever scored
on the data that produced it.

In [ ]:
import json
import numpy as np
from scipy.stats import wilcoxon

# ---------------------------------------------------------------- data
big = [json.loads(l) for l in open("followup_n900.jsonl") if l.strip()]
big.sort(key=lambda z: z["id"])
M900 = ["P_esc alone", "sum_j p_ij", "MIA/PMIA", "IRIE",
        "push PPR rule-a", "escape rule-a", "MC 50", "MC 200"]

HO = json.load(open("holdout.json"))["records"]
NAME100 = {"P_esc alone": "P_esc alone", "sum_j p_ij": "sum_j p_ij",
           "MIA/PMIA": "MIA/PMIA (KDD10)", "IRIE": "IRIE (ICDM12)",
           "push PPR rule-a": "PPR push rule-a", "escape rule-a": "THIS WORK",
           "MC 50": "MC 50 samples", "MC 200": "MC 200 samples"}

small = [dict(mean_pesc=r["mean_pesc"],
              rho={k: r["rho"][v] for k, v in NAME100.items()},
              ms={k: r["ms"][v] for k, v in NAME100.items()}) for r in HO]
for r in big:
    r["_pe"] = r["mean_pesc"]
for r in small:
    r["_pe"] = r["mean_pesc"]


def fit_router(rows, edges, methods):
    """Best method per Pesc bucket, learned on `rows`."""
    pick = {}
    for b in range(len(edges) - 1):
        sel = [x for x in rows if edges[b] <= x["_pe"] < edges[b + 1]]
        if not sel:
            pick[b] = methods[0]
            continue
        pick[b] = max(methods,
                      key=lambda m: np.mean([x["rho"][m] for x in sel]))
    return pick


def apply_router(rows, edges, pick, field="rho"):
    out = []
    for x in rows:
        b = min(np.searchsorted(edges, x["_pe"], side="right") - 1,
                len(edges) - 2)
        b = max(b, 0)
        out.append(x[field][pick[b]])
    return np.array(out)


def kfold(rows, edges, methods, k=5, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(rows))
    folds = np.array_split(idx, k)
    acc, cost = [], []
    for f in folds:
        te = [rows[i] for i in f]
        tr = [rows[i] for i in idx if i not in set(f.tolist())]
        pick = fit_router(tr, edges, methods)
        acc.append(apply_router(te, edges, pick))
        cost.append(apply_router(te, edges, pick, field="ms"))
    return np.concatenate(acc), np.concatenate(cost)


E8 = np.linspace(0, 1, 9)

print("=" * 74)
print("1. k-FOLD CROSS-VALIDATION, n=900, 400 instances, 8 buckets")
print("=" * 74)
r_acc, r_cost = kfold(big, E8, M900, k=5)
print(f"  {'method':22s} {'mean rho':>9s} {'mean ms':>9s}")
singles = {m: np.mean([x["rho"][m] for x in big]) for m in M900}
for m in sorted(M900, key=lambda z: -singles[z]):
    print(f"  {m:22s} {singles[m]:9.4f} "
          f"{np.mean([x['ms'][m] for x in big]):9.2f}")
print(f"  {'ROUTER (5-fold CV)':22s} {r_acc.mean():9.4f} {r_cost.mean():9.2f}")
bestm = max(singles, key=singles.get)
d = r_acc - np.array([x["rho"][bestm] for x in big])
print(f"\n  gain over best single ({bestm}): {r_acc.mean()-singles[bestm]:+.4f}"
      f"  (p={wilcoxon(d[np.abs(d)>1e-12]).pvalue:.2e})")
ceil = np.mean([max(x["rho"][m] for m in M900) for x in big])
print(f"  per-instance oracle SELECTION (ceiling): {ceil:.4f}")
print(f"  router captures {100*(r_acc.mean()-singles[bestm])/(ceil-singles[bestm]):.0f}%"
      f" of the available selection gain")

print("\n" + "=" * 74)
print("2. HOW FEW THRESHOLDS ARE NEEDED?  (5-fold CV each)")
print("=" * 74)
for name, edges in [("1 bucket (no routing)", np.array([0., 1.])),
                    ("2 buckets @0.50", np.array([0., .5, 1.])),
                    ("2 buckets @0.875", np.array([0., .875, 1.])),
                    ("3 buckets @0.625/0.875", np.array([0., .625, .875, 1.])),
                    ("4 buckets", np.linspace(0, 1, 5)),
                    ("8 buckets", E8),
                    ("16 buckets", np.linspace(0, 1, 17))]:
    a, c = kfold(big, edges, M900, k=5)
    print(f"  {name:26s} rho {a.mean():.4f}   ms {c.mean():6.2f}")

print("\n" + "=" * 74)
print("3. CROSS-SCALE TRANSFER OF THE THRESHOLDS")
print("=" * 74)
E3 = np.array([0., .625, .875, 1.])
pick900 = fit_router(big, E3, M900)
pick100 = fit_router(small, E3, M900)
print(f"  learned at n=900 : {[pick900[b] for b in range(3)]}")
print(f"  learned at n=100 : {[pick100[b] for b in range(3)]}")
a_cross = apply_router(small, E3, pick900)
a_native = apply_router(small, E3, pick100)
s100 = {m: np.mean([x["rho"][m] for x in small]) for m in M900}
b100 = max(s100, key=s100.get)
print(f"\n  at n=100, best single ({b100}): {s100[b100]:.4f}")
print(f"  at n=100, router fitted at n=100     : {a_native.mean():.4f}")
print(f"  at n=100, router fitted at n=900     : {a_cross.mean():.4f}  "
      f"<-- transfers without refitting")

print("\n" + "=" * 74)
print("4. WHAT THE ROUTER BUYS, BY STRATUM (n=900, 3-bucket rule)")
print("=" * 74)
print(f"  {'Pesc range':16s} {'chosen':16s} {'rho':>7s} {'ms':>8s} "
      f"{'vs IRIE':>9s}")
for b in range(3):
    sel = [x for x in big if E3[b] <= x["_pe"] < E3[b + 1]]
    m = pick900[b]
    rr = np.mean([x["rho"][m] for x in sel])
    mm = np.mean([x["ms"][m] for x in sel])
    ir = np.mean([x["rho"]["IRIE"] for x in sel])
    print(f"  [{E3[b]:.3f},{E3[b+1]:.3f})   {m:16s} {rr:7.4f} {mm:8.2f} "
          f"{rr-ir:+9.4f}")

json.dump(dict(kfold_rho=float(r_acc.mean()), kfold_ms=float(r_cost.mean()),
               best_single=bestm, best_single_rho=float(singles[bestm]),
               ceiling=float(ceil),
               pick3={str(k): v for k, v in pick900.items()},
               cross_scale=dict(n100_best=float(s100[b100]),
                                n100_router_native=float(a_native.mean()),
                                n100_router_from900=float(a_cross.mean()))),
          open("router.json", "w"), indent=1)
print("\nwrote router.json")

**Result.** Three buckets — IRIE below $0.625$, Monte Carlo to $0.875$, the
raw escape score above — reach $\rho=0.948$ against $0.917$ for the best single
method ($p=0.009$), capture 59% of the per-instance oracle selection gain, and
cost less than the sampling baseline they sometimes call. Thresholds fitted at
$n=900$ transfer to $n=100$ unchanged.

The router knows nothing the individual methods do not. It exploits the fact
that their failure modes are **disjoint and predictable**, and that
$\bar P_{esc}$ measures the thing that determines which failure is active.

## 10. Diagnostics: three claims put to the test

Three statements made elsewhere were asserted rather than measured. One of them
turns out to be wrong.

In [ ]:
import json
import numpy as np
from scipy.stats import wilcoxon, pearsonr
from escape_lib import (make_instance, ground_truth, rho, diffuse,
                        alpha_from_rule, m_escape_pull)

CAL = json.load(open("calibration.json"))
COEF = (CAL["intercept"], CAL["slope"])
AT = json.load(open("alpha_transfer.json"))
PUSH_COEF = tuple(AT["rules"]["PPR push"])
GRID = np.round(np.arange(0.40, 1.001, 0.025), 4)

TOPOS = ["er", "ba", "grid", "ws"]
DEGS = [4, 6, 8, 10, 12]
PRIORS = ["dirichlet", "uniform", "sparse"]


def stratified_sample(n, per_bin, n_bins=8, seed=7):
    """Same design as the n=900 study, at arbitrary n."""
    rng = np.random.default_rng(seed)
    bins = [[] for _ in range(n_bins)]
    tries = 0
    while sum(len(b) for b in bins) < n_bins * per_bin and tries < 200000:
        tries += 1
        topo = TOPOS[rng.integers(len(TOPOS))]
        deg = int(DEGS[rng.integers(len(DEGS))])
        p_lo = float(np.exp(rng.uniform(np.log(0.005), np.log(0.65))))
        p_hi = float(min(p_lo + rng.uniform(0.02, 0.30), 0.97))
        prior = PRIORS[rng.integers(len(PRIORS))]
        sd = int(rng.integers(1, 10**6))
        inst = make_instance(topo, n, p_lo, p_hi, sd, prior_kind=prior,
                             avg_deg=deg)
        b = min(int(inst.mean_pesc * n_bins), n_bins - 1)
        if len(bins[b]) >= per_bin:
            continue
        bins[b].append((topo, deg, p_lo, p_hi, prior, sd))
    return [c for b in bins for c in b], tries


# ===================================================================== A
print("=" * 78)
print("A. SCALE vs DESIGN: the same stratified design, run at n=100")
print("=" * 78)
cfgs, tries = stratified_sample(100, 25)
print(f"   {len(cfgs)} instances (8 strata x 25), {tries} draws\n")

esc, push, pes = [], [], []
for (topo, deg, lo, hi, prior, sd) in cfgs:
    ins = make_instance(topo, 100, lo, hi, sd, prior_kind=prior, avg_deg=deg)
    gt = ground_truth(ins, 3000, seed=sd % 10**6 + 3)
    a_e = alpha_from_rule(ins, COEF)
    a_p = float(np.clip(PUSH_COEF[0] + PUSH_COEF[1] * ins.mean_pesc, 0.4, 1.0))
    esc.append(rho(diffuse(ins, a_e, use_pesc=True, orientation="pull"), gt))
    push.append(rho(diffuse(ins, a_p, use_pesc=False, orientation="push"), gt))
    pes.append(ins.mean_pesc)
esc, push = np.array(esc), np.array(push)
d = esc - push
nz = d[np.abs(d) > 1e-12]
print(f"   escape-scaled rule-a : {esc.mean():.4f}")
print(f"   push PPR   rule-a    : {push.mean():.4f}")
print(f"   difference           : {d.mean():+.4f}  "
      f"(escape wins {(d>0).sum()}/{len(d)}, p={wilcoxon(nz).pvalue:.2e})")
print(f"\n   for reference, grid design at n=100 : escape 0.894, push 0.904 "
      f"(p=0.60)")
print(f"   stratified design at n=900          : escape 0.910, push 0.857 "
      f"(p=1.5e-21)")
verdict = ("DESIGN, not scale" if wilcoxon(nz).pvalue < 0.05 and d.mean() > 0
           else "SCALE, not design")
print(f"\n   >>> the gap is driven by: {verdict}")

# ===================================================================== B
print("\n" + "=" * 78)
print("B. IS THE rho-vs-alpha OPTIMUM FLATTER AT n=900?")
print("=" * 78)


def plateau_width(curve, grid, eps=0.005):
    """Total width of alpha values within eps of the instance's own maximum."""
    c = np.asarray(curve)
    good = c >= c.max() - eps
    return float(good.sum() / len(good))          # fraction of the grid


# n = 64 : recompute curves on the calibration instances
CAL_TOPOS = ["er", "ba", "grid"]
CAL_RANGES = [(0.01, 0.05), (0.03, 0.10), (0.07, 0.20),
              (0.15, 0.35), (0.25, 0.50), (0.40, 0.70)]
w64 = []
for topo in CAL_TOPOS:
    for (lo, hi) in CAL_RANGES:
        for sd in [11, 12, 13]:
            ins = make_instance(topo, 64, lo, hi, sd)
            gt = ground_truth(ins, 2500, seed=1000 + sd)
            c = [rho(diffuse(ins, float(a), use_pesc=True, orientation="pull"),
                     gt) for a in GRID]
            w64.append(plateau_width(c, GRID))

fu = [json.loads(l) for l in open("followup_n900.jsonl") if l.strip()]
w900 = [plateau_width(r["curve_escape"], GRID) for r in fu]

print(f"   fraction of the alpha grid within 0.005 of the instance optimum")
print(f"     n=64  ({len(w64)} instances) : {np.mean(w64):.3f} "
      f"(median {np.median(w64):.3f})")
print(f"     n=900 ({len(w900)} instances): {np.mean(w900):.3f} "
      f"(median {np.median(w900):.3f})")
ratio = np.mean(w900) / np.mean(w64)
print(f"     ratio n=900 / n=64 : {ratio:.2f}x")
print(f"\n   >>> the 'flatter optimum' explanation is "
      f"{'SUPPORTED' if ratio > 1.15 else 'NOT SUPPORTED'}")

# spread of rho across the whole grid, as a second view
sp64 = np.mean([1.0] * 0) if False else None
print("\n   secondary view: how much rho is at stake across the grid")
print(f"     n=900 mean (max - min) over alpha : "
      f"{np.mean([max(r['curve_escape'])-min(r['curve_escape']) for r in fu]):.3f}")

# ===================================================================== C
print("\n" + "=" * 78)
print("C. SENSITIVITY OF THE RULE TO ITS COEFFICIENTS (held-out, n=100)")
print("=" * 78)
HO = json.load(open("holdout.json"))
base_cfgs = [(r["topology"], r["lo"], r["hi"], r["seed"]) for r in HO["records"]]
insts = []
for (topo, lo, hi, sd) in base_cfgs:
    ins = make_instance(topo, 100, lo, hi, sd)
    insts.append((ins, ground_truth(ins, 3000, seed=5000 + sd)))

a0, b0 = COEF
print(f"   fitted rule: clip({a0:.3f} + {b0:.3f} Pesc, 0.4, 1)")
print(f"   {'perturbation':28s} {'mean rho':>9s} {'delta':>8s}")
base = np.mean([rho(m_escape_pull(i, alpha=float(np.clip(a0 + b0 * i.mean_pesc,
                                                         .4, 1))), g)
                for i, g in insts])
print(f"   {'(none)':28s} {base:9.4f} {0.0:+8.4f}")
sens = {}
for lbl, (da, db) in [("intercept -0.10", (-0.10, 0.0)),
                      ("intercept +0.10", (+0.10, 0.0)),
                      ("slope -0.20", (0.0, -0.20)),
                      ("slope +0.20", (0.0, +0.20)),
                      ("both -10% relative", (-0.1 * a0, -0.1 * b0)),
                      ("both +10% relative", (+0.1 * a0, +0.1 * b0)),
                      ("rounded to 0.5+0.6", (0.5 - a0, 0.6 - b0))]:
    aa, bb = a0 + da, b0 + db
    v = np.mean([rho(m_escape_pull(i, alpha=float(np.clip(aa + bb * i.mean_pesc,
                                                          .4, 1))), g)
                 for i, g in insts])
    sens[lbl] = float(v)
    print(f"   {lbl:28s} {v:9.4f} {v-base:+8.4f}")

json.dump(dict(scale_vs_design=dict(n100_strat_escape=float(esc.mean()),
                                    n100_strat_push=float(push.mean()),
                                    diff=float(d.mean()),
                                    wins=int((d > 0).sum()), n=len(d),
                                    p=float(wilcoxon(nz).pvalue)),
               plateau=dict(n64=float(np.mean(w64)), n900=float(np.mean(w900)),
                            ratio=float(ratio)),
               sensitivity=dict(base=float(base), **sens)),
          open("diagnostics.json", "w"), indent=1)
print("\nwrote diagnostics.json")

**A — the claim was wrong.** Under the *same* stratified design at $n=100$
the escape-scaled operator still leads push PPR ($+0.040$,
$p=1.7\times10^{-10}$). The separation comes from balanced coverage of the
density axis, **not** from scale. The fixed grid under-samples the regimes
where sub-stochastic scaling pays.

**B — the claim holds, with a number.** The fraction of the $\alpha$ grid
within $0.005$ of an instance's optimum grows from $0.147$ at $n=64$ to $0.290$
at $n=900$: the plateau is $1.97\times$ wider, so a noisier argmax with
undiminished practical benefit is exactly what to expect.

**C — robust to rounding, asymmetric in direction.** $0.5+0.6\bar P_{esc}$
costs only $0.009$. But $+0.10$ on the intercept costs $0.031$ while $-0.10$
costs $0.199$. Err upward.

## 11. Per-node damping: a real ceiling, and a located negative result

Would a per-vertex $\alpha_i$ help? Find the **ceiling** first — optimise the
$\alpha$ vector directly against ground truth. If that does not clear the
scalar oracle, no rule can. Then test whether $P_{esc}$-based features reach
it.

In [ ]:
import json
import numpy as np
from scipy.stats import wilcoxon
from escape_lib import (make_instance, ground_truth, rho, alpha_from_rule,
                        m_escape_pull, oracle_alpha)

CAL = json.load(open("calibration.json"))
COEF = (CAL["intercept"], CAL["slope"])


# ---------------------------------------------------------------- machinery
def smoothed_pesc(inst, beta, weighting="log", iters=200, tol=1e-13):
    """S = (1-beta) Pesc + beta W S : escape probability diffused over
    the neighbourhood, decaying with distance at rate beta."""
    if beta <= 0:
        return inst.pesc.copy()
    W = inst.W(weighting)
    pe = inst.pesc
    S = pe.copy()
    for _ in range(iters):
        Sn = (1.0 - beta) * pe + beta * (W @ S)
        if np.abs(Sn - S).max() < tol:
            return Sn
        S = Sn
    return S


def diffuse_vec(inst, a, weighting="log", tol=1e-13, max_iter=800):
    """Escape-scaled pull iteration with a per-node damping VECTOR a."""
    W = inst.W(weighting)
    pe = inst.pesc
    x = inst.x
    a = np.asarray(a, dtype=float)
    y = x.copy()
    for _ in range(max_iter):
        z = pe * (W @ y)
        yn = (1.0 - a) * x + a * z
        s = yn.sum()
        if s > 0:
            yn = yn / s
        if np.abs(yn - y).max() < tol:
            return yn
        y = yn
    return y


# =================================================== STEP 1: the ceiling
print("=" * 76)
print("STEP 1  CEILING: can a per-node alpha beat the scalar oracle at all?")
print("=" * 76)

TOPOS = ["er", "ba", "grid", "ws"]
RANGES = [(0.02, 0.08), (0.18, 0.38), (0.45, 0.72)]
SEEDS = [101, 102]
CAND = np.round(np.arange(0.40, 1.001, 0.075), 4)

sc_or, pn_or, sc_rule = [], [], []
for topo in TOPOS:
    for (lo, hi) in RANGES:
        for sd in SEEDS:
            ins = make_instance(topo, 100, lo, hi, sd)
            gt = ground_truth(ins, 3000, seed=5000 + sd)

            a_scalar, r_scalar = oracle_alpha(ins, gt)
            sc_or.append(r_scalar)
            sc_rule.append(rho(m_escape_pull(ins, alpha=alpha_from_rule(ins, COEF)), gt))

            # coordinate ascent on the alpha vector, warm-started at the
            # scalar oracle; 3 sweeps over all vertices
            a = np.full(ins.n, a_scalar)
            best = rho(diffuse_vec(ins, a), gt)
            for _ in range(3):
                improved = False
                for i in range(ins.n):
                    keep, cur = a[i], best
                    for c in CAND:
                        if c == keep:
                            continue
                        a[i] = c
                        r = rho(diffuse_vec(ins, a), gt)
                        if r > cur + 1e-9:
                            cur, keep = r, c
                    a[i] = keep
                    if cur > best + 1e-9:
                        best, improved = cur, True
                if not improved:
                    break
            pn_or.append(best)
    print("  ceiling swept", topo, flush=True)

sc_or, pn_or, sc_rule = map(np.array, (sc_or, pn_or, sc_rule))
head = pn_or - sc_or
print(f"\n  instances                       : {len(sc_or)}")
print(f"  scalar rule  (what we ship)     : {sc_rule.mean():.4f}")
print(f"  scalar ORACLE                   : {sc_or.mean():.4f}")
print(f"  per-node ORACLE (coord. ascent) : {pn_or.mean():.4f}")
print(f"  headroom over scalar oracle     : {head.mean():+.4f} "
      f"(max {head.max():+.4f}, wins {(head>1e-6).sum()}/{len(head)})")
print(f"  headroom over the shipped rule  : {(pn_or-sc_rule).mean():+.4f}")

# =================================================== STEP 2: rule families
print("\n" + "=" * 76)
print("STEP 2  RULE FAMILIES: fit c0,c1 per family on the calibration set")
print("=" * 76)

CAL_TOPOS = ["er", "ba", "grid"]
CAL_RANGES = [(0.01, 0.05), (0.03, 0.10), (0.07, 0.20),
              (0.15, 0.35), (0.25, 0.50), (0.40, 0.70)]
CAL_SEEDS = [11, 12, 13]
cal = []
for topo in CAL_TOPOS:
    for (lo, hi) in CAL_RANGES:
        for sd in CAL_SEEDS:
            ins = make_instance(topo, 64, lo, hi, sd)
            cal.append((ins, ground_truth(ins, 2500, seed=1000 + sd)))
print(f"  calibration instances: {len(cal)}")

BETAS = [None, 0.0, 0.25, 0.5, 0.75, 0.9]     # None = global mean (current)
C0 = np.round(np.arange(0.20, 0.91, 0.10), 3)
C1 = np.round(np.arange(0.00, 1.01, 0.10), 3)


def stat_for(inst, beta):
    if beta is None:
        return np.full(inst.n, inst.mean_pesc)
    return smoothed_pesc(inst, beta)


fitted = {}
for beta in BETAS:
    pre = [(i, g, stat_for(i, beta)) for i, g in cal]
    best, bc = -2.0, None
    for c0 in C0:
        for c1 in C1:
            tot = 0.0
            for i, g, S in pre:
                a = np.clip(c0 + c1 * S, 0.4, 1.0)
                tot += rho(diffuse_vec(i, a), g)
            m = tot / len(pre)
            if m > best:
                best, bc = m, (float(c0), float(c1))
    fitted[str(beta)] = dict(c0=bc[0], c1=bc[1], cal_rho=float(best))
    lab = "global mean" if beta is None else f"beta={beta}"
    print(f"  {lab:14s} -> a=clip({bc[0]:.2f}+{bc[1]:.2f}*S)  "
          f"calibration rho {best:.4f}")

# ---- evaluate all families out of sample
print("\n" + "=" * 76)
print("HELD-OUT, n=100, 100 instances")
print("=" * 76)
HO = json.load(open("holdout.json"))
insts = []
for r in HO["records"]:
    ins = make_instance(r["topology"], 100, r["lo"], r["hi"], r["seed"])
    insts.append((ins, ground_truth(ins, 3000, seed=5000 + r["seed"])))

res = {}
for beta in BETAS:
    c0, c1 = fitted[str(beta)]["c0"], fitted[str(beta)]["c1"]
    vals = []
    for ins, g in insts:
        a = np.clip(c0 + c1 * stat_for(ins, beta), 0.4, 1.0)
        vals.append(rho(diffuse_vec(ins, a), g))
    res[str(beta)] = np.array(vals)

ship = np.array([rho(m_escape_pull(i, alpha=alpha_from_rule(i, COEF)), g)
                 for i, g in insts])
res["shipped"] = ship

base = res["None"]
print(f"  {'family':22s} {'mean rho':>9s} {'vs global':>10s} {'wins':>9s} {'p':>10s}")
for key in ["shipped", "None", "0.0", "0.25", "0.5", "0.75", "0.9"]:
    v = res[key]
    lab = {"shipped": "shipped scalar rule", "None": "refit global mean",
           "0.0": "per-node, local Pesc"}.get(key, f"per-node, beta={key}")
    if key == "None":
        print(f"  {lab:22s} {v.mean():9.4f} {'--':>10s} {'--':>9s} {'--':>10s}")
        continue
    d = v - base
    nz = d[np.abs(d) > 1e-12]
    p = wilcoxon(nz).pvalue if len(nz) else 1.0
    print(f"  {lab:22s} {v.mean():9.4f} {d.mean():+10.4f} "
          f"{(d>0).sum():4d}/100 {p:10.2e}")

json.dump(dict(ceiling=dict(scalar_rule=float(sc_rule.mean()),
                            scalar_oracle=float(sc_or.mean()),
                            pernode_oracle=float(pn_or.mean()),
                            headroom=float(head.mean()),
                            n=len(sc_or)),
               fitted=fitted,
               heldout={k: float(v.mean()) for k, v in res.items()}),
          open("pernode.json", "w"), indent=1)
print("\nwrote pernode.json")

In [ ]:
# Honest ceiling: fit the alpha vector on one ground truth, score on another
exec(open("run_pernode_honest.py").read().split('"""', 2)[-1])

**The ceiling is real** — $+0.062$ on an *independent* ground truth, only
11% of the apparent gain lost to overfitting, concentrated in the dense regime
where the diffusions currently fail (up to $+0.341$ on dense grids).

**But $P_{esc}$ does not reach it.** Setting $\alpha_i$ from neighbourhood-
smoothed $P_{esc}$ gets monotonically *worse* the more local it becomes
(0.867 purely local → 0.894 nearly global), because $P_{esc}$ is already used
multiplicatively inside the operator and using it twice is redundant.

The diagnostic is the grid family: all $P_{esc}(i)$ are nearly equal there, so
a local rule has no variation to exploit — yet the per-node oracle still gains
$+0.341$. **The exploitable per-vertex signal is positional or structural, not
escape-based.** That is where to dig next, and where not to.

## 12. Figures

In [ ]:
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CAL = json.load(open("calibration.json"))
AT = json.load(open("alpha_transfer.json"))
HO = json.load(open("holdout.json"))
SI = json.load(open("scale_identity.json"))
SW = json.load(open("alpha_sweep.json"))

plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": 0.3,
                     "figure.dpi": 160})
fig, axes = plt.subplots(2, 2, figsize=(11.0, 7.4))
ax = axes.ravel()

# ---------------- A: calibration -------------------------------------------
pe = np.array([r["mean_pesc"] for r in CAL["rows"]])
al = np.array([r["alpha_opt"] for r in CAL["rows"]])
ax[0].scatter(pe, al, s=30, alpha=0.65, edgecolor="k", linewidth=0.4,
              zorder=3, label="optimal $\\alpha$ per training instance")
g = np.linspace(0, 1, 200)
a0, b0 = CAL["intercept"], CAL["slope"]
ap, bp = AT["rules"]["PPR push"]
ax[0].plot(g, np.clip(a0 + b0 * g, 0.4, 1.0), "r-", lw=2.2,
           label=rf"escape-scaled: ${a0:.3f}+{b0:.3f}\bar{{P}}_{{esc}}$"
                 f"\n   $r={CAL['pearson']:+.3f}$, RMSE ${CAL['rmse']:.3f}$")
ax[0].plot(g, np.clip(ap + bp * g, 0.4, 1.0), "--", color="tab:green", lw=2.2,
           label=rf"push PPR: ${ap:.3f}+{bp:.3f}\bar{{P}}_{{esc}}$"
                 "\n   $r=+0.961$, RMSE $0.068$")
ax[0].set_xlabel(r"mean escape probability $\bar{P}_{\mathrm{esc}}$")
ax[0].set_ylabel(r"empirically optimal $\alpha$")
ax[0].set_title("(a) Calibration on 54 training instances", fontsize=10)
ax[0].legend(fontsize=7, loc="lower right")
ax[0].set_ylim(0.42, 1.07)

# ---------------- B: out-of-sample alpha sweep ------------------------------
gr = np.array(SW["grid"])
sty = {"escape-scaled pull": ("tab:orange", "-"), "push PPR": ("tab:green", "--")}
for fam, (c, ls) in sty.items():
    d = SW[fam]
    ax[1].plot(gr, d["curve"], ls, color=c, lw=1.9, label=f"{fam}: fixed $\\alpha$")
    ax[1].axhline(d["rule_rho"], color=c, lw=1.1, alpha=0.55, ls=":")
    ax[1].plot([d["best_fixed_alpha"]], [d["best_fixed"]], "v", color=c, ms=7,
               markeredgecolor="k", markeredgewidth=0.4)
    ax[1].annotate(f"{fam}, rule-$\\alpha$: {d['rule_rho']:.3f}",
                   xy=(0.995, d["rule_rho"]), fontsize=7, color=c,
                   va="bottom", ha="right")
ax[1].set_xlabel(r"damping factor $\alpha$ (single value applied to all instances)")
ax[1].set_ylabel(r"mean Spearman $\rho$")
ax[1].set_title("(b) No fixed $\\alpha$ matches the per-instance rule", fontsize=10)
ax[1].legend(fontsize=7, loc="lower right", framealpha=0.93)
ax[1].set_ylim(0.32, 0.99)

# ---------------- C: baselines ---------------------------------------------
recs = HO["records"]
push_rule = float(np.mean(AT["results"]["PPR push|rule"]))
show = [("x only", "base", "$x$ only"),
        ("PPR pull rule-a", "base", "PPR pull"),
        ("P_esc alone", "base", r"$P_{\mathrm{esc}}$"),
        ("sum_j p_ij", "base", r"$\sum_j p_{ij}$"),
        ("MC 10 samples", "samp", "MC 10"),
        ("PPR push a=.85", "base", r"push PPR $\alpha{=}.85$"),
        ("MIA/PMIA (KDD10)", "base", "MIA/PMIA"),
        ("MC 50 samples", "samp", "MC 50"),
        ("THIS WORK", "ours", "escape-scaled rule-$\\alpha$"),
        ("__push_rule__", "ours", "push PPR rule-$\\alpha$"),
        ("oracle a", "orac", "oracle $\\alpha$"),
        ("IRIE (ICDM12)", "base", "IRIE"),
        ("MC 200 samples", "samp", "MC 200")]
vals = [push_rule if k == "__push_rule__"
        else float(np.mean([r["rho"][k] for r in recs])) for k, _, _ in show]
order = np.argsort(vals)
cols = {"base": "0.74", "samp": "tab:blue", "ours": "tab:orange", "orac": "0.40"}
for pos, i in enumerate(order):
    ax[2].bar(pos, vals[i], color=cols[show[i][1]], edgecolor="k", linewidth=0.5)
    ax[2].text(pos, vals[i] + 0.014, f"{vals[i]:.3f}", ha="center", fontsize=6.3)
ax[2].set_xticks(range(len(show)))
ax[2].set_xticklabels([show[i][2] for i in order],
                      fontsize=6.3, rotation=45, ha="right")
ax[2].set_ylabel(r"mean Spearman $\rho$")
ax[2].set_ylim(0, 1.06)
ax[2].set_title(r"(c) Out-of-sample comparison, $n=100$, 100 instances", fontsize=10)
h = [plt.Rectangle((0, 0), 1, 1, fc=cols[c], ec="k", lw=0.5)
     for c in ("base", "samp", "ours", "orac")]
ax[2].legend(h, ["heuristic baseline", "sampling estimator",
                 "calibrated by our rule", "per-instance oracle"],
             fontsize=6.4, loc="upper left", framealpha=0.93)

# ---------------- D: cost/accuracy -----------------------------------------
mc = SI["mc_curve"]
bs = sorted(int(b) for b in mc)
ax[3].plot([mc[str(b)]["ms"] for b in bs], [mc[str(b)]["rho"] for b in bs],
           "o-", color="tab:blue", ms=4, lw=1.5, label="Monte Carlo (5-320 samples)")
for b in bs:
    if b in (5, 20, 80, 320):
        ax[3].annotate(f"{b}", (mc[str(b)]["ms"], mc[str(b)]["rho"]),
                       textcoords="offset points", xytext=(3, -9),
                       fontsize=6, color="tab:blue")
o = SI["ours_small"]
ax[3].plot([o["ms"]], [o["rho"]], "*", color="tab:orange", ms=17,
           markeredgecolor="k", markeredgewidth=0.5, zorder=5,
           label=f"escape-scaled rule-$\\alpha$ ({o['rho']:.3f})")
n100 = {k: float(np.mean([r["rho"][k] for r in recs]))
        for k in ("IRIE (ICDM12)", "MIA/PMIA (KDD10)", "P_esc alone")}
t100 = {k: float(np.mean([r["ms"][k] for r in recs])) for k in n100}
for k, mk, c, lb in [("IRIE (ICDM12)", "s", "tab:green", "IRIE"),
                     ("MIA/PMIA (KDD10)", "^", "tab:red", "MIA/PMIA"),
                     ("P_esc alone", "D", "0.4", r"$P_{\mathrm{esc}}$ alone")]:
    ax[3].plot([t100[k]], [n100[k]], mk, color=c, ms=6.5,
               markeredgecolor="k", markeredgewidth=0.4, label=lb)
ax[3].set_xscale("log")
ax[3].set_xlabel("wall-clock per instance (ms, log scale)")
ax[3].set_ylabel(r"mean Spearman $\rho$")
ax[3].set_title("(d) Cost / accuracy", fontsize=10)
ax[3].legend(fontsize=6.6, loc="lower right", framealpha=0.93)

plt.tight_layout()
plt.savefig("fig1.pdf", bbox_inches="tight")
plt.savefig("fig1.png", bbox_inches="tight", dpi=150)
print("wrote fig1.pdf / fig1.png")

In [ ]:
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CFG = json.load(open("followup_configs.json"))
recs = [json.loads(l) for l in open("followup_n900.jsonl") if l.strip()]
recs.sort(key=lambda r: r["id"])
GRID = np.array(CFG["alpha_grid"])
NB = CFG["n_bins"]

plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": 0.3,
                     "figure.dpi": 160})
fig, axes = plt.subplots(2, 2, figsize=(11.0, 7.4))
ax = axes.ravel()

pe = np.array([r["mean_pesc"] for r in recs])
bins = np.array([r["bin"] for r in recs])
centers = np.array([(b + 0.5) / NB for b in range(NB)])

# --------------- (a) stratified performance --------------------------------
SHOW = [("escape rule-a", "tab:orange", "-", "o", "escape-scaled rule-$\\alpha$"),
        ("IRIE", "tab:green", "--", "s", "IRIE"),
        ("MIA/PMIA", "tab:red", "-.", "^", "MIA/PMIA"),
        ("push PPR rule-a", "tab:purple", ":", "v", "push PPR rule-$\\alpha$"),
        ("MC 200", "tab:blue", "-", "d", "Monte Carlo, 200"),
        ("P_esc alone", "0.35", "--", "x", r"$P_{\mathrm{esc}}$ alone")]
for key, c, ls, mk, lb in SHOW:
    ys = [np.mean([r["rho"][key] for r in recs if r["bin"] == b])
          for b in range(NB)]
    ax[0].plot(centers, ys, ls, color=c, marker=mk, ms=4.5, lw=1.6, label=lb)
ax[0].set_xlabel(r"mean escape probability $\bar{P}_{\mathrm{esc}}$ (stratum centre)")
ax[0].set_ylabel(r"mean Spearman $\rho$")
ax[0].set_title("(a) Performance by density stratum (50 instances each)",
                fontsize=10)
ax[0].legend(fontsize=6.6, loc="lower left", ncol=2, framealpha=0.93)
ax[0].set_ylim(0.15, 1.03)

# --------------- (b) alpha* vs Pesc at n=900 -------------------------------
ao = np.array([r["alpha_opt_escape"] for r in recs])
ax[1].scatter(pe, ao, s=11, alpha=0.35, edgecolor="none", color="tab:orange",
              label=r"per-instance optimal $\alpha$, $n{=}900$")
g = np.linspace(0, 1, 200)
a64, b64 = CFG["coef_escape"]
b900, a900 = np.polyfit(pe, ao, 1)
ax[1].plot(g, np.clip(a64 + b64 * g, 0.4, 1.0), "r-", lw=2.2,
           label=rf"rule fitted at $n{{=}}64$: ${a64:.3f}+{b64:.3f}\bar{{P}}_{{esc}}$")
ax[1].plot(g, np.clip(a900 + b900 * g, 0.4, 1.0), "--", color="k", lw=1.8,
           label=rf"refit at $n{{=}}900$: ${a900:.3f}+{b900:.3f}\bar{{P}}_{{esc}}$")
med = [np.median(ao[bins == b]) for b in range(NB)]
ax[1].plot(centers, med, "o-", color="darkorange", ms=6, lw=2,
           markeredgecolor="k", markeredgewidth=0.5, label="per-stratum median")
ax[1].set_xlabel(r"mean escape probability $\bar{P}_{\mathrm{esc}}$")
ax[1].set_ylabel(r"empirically optimal $\alpha$")
ax[1].set_title(r"(b) The rule extrapolates from $n{=}64$ to $n{=}900$",
                fontsize=10)
ax[1].legend(fontsize=6.5, loc="lower right", framealpha=0.93)
ax[1].set_ylim(0.37, 1.06)

# --------------- (c) rule vs fixed vs oracle -------------------------------
lbl, rule_v, fix_v, orc_v = [], [], [], []
for fam, ck, ok, rk in [("escape-scaled", "curve_escape", "oracle_escape",
                         "escape rule-a"),
                        ("push PPR", "curve_push", "oracle_push",
                         "push PPR rule-a")]:
    mc = np.array([r[ck] for r in recs]).mean(axis=0)
    lbl.append(fam)
    fix_v.append(float(mc.max()))
    rule_v.append(float(np.mean([r["rho"][rk] for r in recs])))
    orc_v.append(float(np.mean([r[ok] for r in recs])))
xp = np.arange(2)
w = 0.26
ax[2].bar(xp - w, fix_v, w, color="0.75", edgecolor="k", lw=0.5,
          label=r"best fixed $\alpha$ (hindsight)")
ax[2].bar(xp, rule_v, w, color="tab:orange", edgecolor="k", lw=0.5,
          label=r"rule-$\alpha$ (fitted at $n{=}64$)")
ax[2].bar(xp + w, orc_v, w, color="0.40", edgecolor="k", lw=0.5,
          label="per-instance oracle")
for i in range(2):
    for off, v in ((-w, fix_v[i]), (0, rule_v[i]), (w, orc_v[i])):
        ax[2].text(i + off, v + 0.008, f"{v:.3f}", ha="center", fontsize=6.6)
ax[2].set_xticks(xp)
ax[2].set_xticklabels(lbl, fontsize=8.5)
ax[2].set_ylabel(r"mean Spearman $\rho$")
ax[2].set_ylim(0, 1.05)
ax[2].set_title(r"(c) The rule beats any constant $\alpha$, $n{=}900$",
                fontsize=10)
ax[2].legend(fontsize=6.8, loc="lower right", framealpha=0.93)

# --------------- (d) paired differences ------------------------------------
ref = np.array([r["rho"]["escape rule-a"] for r in recs])
cmp = [("IRIE", "tab:green"), ("MC 200", "tab:blue"),
       ("MIA/PMIA", "tab:red"), ("push PPR rule-a", "tab:purple")]
data = [ref - np.array([r["rho"][k] for r in recs]) for k, _ in cmp]
bp = ax[3].boxplot(data, vert=False, widths=0.6, showfliers=False,
                   patch_artist=True, medianprops=dict(color="k", lw=1.4))
for patch, (_, c) in zip(bp["boxes"], cmp):
    patch.set_facecolor(c)
    patch.set_alpha(0.55)
ax[3].axvline(0, color="k", lw=1.2, ls="--")
for i, d in enumerate(data):
    ax[3].plot([d.mean()], [i + 1], "D", color="k", ms=5, zorder=5)
ax[3].set_yticklabels(["IRIE", "MC 200", "MIA/PMIA", "push PPR\nrule-$\\alpha$"],
                      fontsize=7.5)
ax[3].set_xlabel(r"paired difference in $\rho$  (escape rule-$\alpha$ minus baseline)")
ax[3].set_title("(d) Paired differences, 400 instances", fontsize=10)
ax[3].text(0.985, 0.05, "diamond = mean, line = median",
           transform=ax[3].transAxes, fontsize=6.5, ha="right",
           bbox=dict(fc="white", ec="0.7", lw=0.5, alpha=0.9))

plt.tight_layout()
plt.savefig("fig2.pdf", bbox_inches="tight")
plt.savefig("fig2.png", bbox_inches="tight", dpi=150)
print("wrote fig2.pdf / fig2.png")

In [ ]:
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

big = [json.loads(l) for l in open("followup_n900.jsonl") if l.strip()]
big.sort(key=lambda z: z["id"])
R = json.load(open("router.json"))
M = ["P_esc alone", "MIA/PMIA", "IRIE", "push PPR rule-a", "escape rule-a",
     "MC 50", "MC 200"]
NB = 8

plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": 0.3,
                     "figure.dpi": 160})
fig, axes = plt.subplots(2, 2, figsize=(11.0, 7.4))
ax = axes.ravel()
centers = np.array([(b + .5) / NB for b in range(NB)])

STY = {"IRIE": ("tab:green", "s"), "MC 200": ("tab:blue", "d"),
       "P_esc alone": ("0.35", "x"), "escape rule-a": ("tab:orange", "o"),
       "MIA/PMIA": ("tab:red", "^"), "push PPR rule-a": ("tab:purple", "v"),
       "MC 50": ("tab:cyan", "*")}
LBL = {"P_esc alone": r"$P_{\mathrm{esc}}$ alone", "MIA/PMIA": "MIA/PMIA",
       "IRIE": "IRIE", "push PPR rule-a": "push PPR rule-$\\alpha$",
       "escape rule-a": "escape rule-$\\alpha$", "MC 50": "MC 50",
       "MC 200": "MC 200"}

# ---- (a) decision map -----------------------------------------------------
for m in M:
    ys = [np.mean([x["rho"][m] for x in big if x["bin"] == b]) for b in range(NB)]
    c, mk = STY[m]
    ax[0].plot(centers, ys, marker=mk, ms=4.5, lw=1.4, color=c, alpha=.85,
               label=LBL[m])
for lo, hi, nm in [(0, .625, "IRIE"), (.625, .875, "MC 200"),
                   (.875, 1.0, r"$P_{\mathrm{esc}}$")]:
    ax[0].axvspan(lo, hi, color="gold", alpha=.13)
    ax[0].text((lo + hi) / 2, 0.20, nm, ha="center", fontsize=7.5,
               style="italic")
for v in (.625, .875):
    ax[0].axvline(v, color="k", lw=1.0, ls="--", alpha=.6)
ax[0].set_xlabel(r"mean escape probability $\bar{P}_{\mathrm{esc}}$")
ax[0].set_ylabel(r"mean Spearman $\rho$")
ax[0].set_title("(a) Which method wins where, and the routing rule", fontsize=10)
ax[0].legend(fontsize=6.2, ncol=2, loc="lower left", framealpha=.93)
ax[0].set_ylim(0.13, 1.03)

# ---- (b) accuracy vs cost -------------------------------------------------
for m in M:
    r = np.mean([x["rho"][m] for x in big])
    t = np.mean([x["ms"][m] for x in big])
    c, mk = STY[m]
    ax[1].plot([t], [r], mk, color=c, ms=8, markeredgecolor="k",
               markeredgewidth=.4, label=LBL[m])
ax[1].plot([R["kfold_ms"]], [R["kfold_rho"]], "*", color="crimson", ms=20,
           markeredgecolor="k", markeredgewidth=.6, zorder=6,
           label=f"ROUTER ({R['kfold_rho']:.3f})")
ax[1].axhline(R["ceiling"], color="0.3", ls=":", lw=1.3)
ax[1].text(0.30, R["ceiling"] + .003, "per-instance oracle selection",
           fontsize=6.6, color="0.3")
ax[1].set_xscale("log")
ax[1].set_xlabel("mean wall clock per instance (ms, log scale)")
ax[1].set_ylabel(r"mean Spearman $\rho$")
ax[1].set_title("(b) The router is above the accuracy/cost frontier",
                fontsize=10)
ax[1].legend(fontsize=6.4, loc="lower right", framealpha=.93)
ax[1].set_ylim(0.50, 1.0)

# ---- (c) how many buckets -------------------------------------------------
names = ["none", "2 @.50", "2 @.875", "3 @.625/.875", "4", "8", "16"]
vals = [0.9166, 0.9360, 0.9350, 0.9477, 0.9360, 0.9477, 0.9452]
cols = ["0.75"] * len(vals)
cols[3] = "crimson"
ax[2].bar(range(len(vals)), vals, color=cols, edgecolor="k", lw=.5)
for i, v in enumerate(vals):
    ax[2].text(i, v + .002, f"{v:.3f}", ha="center", fontsize=6.5)
ax[2].axhline(R["ceiling"], color="0.3", ls=":", lw=1.3)
ax[2].set_xticks(range(len(names)))
ax[2].set_xticklabels(names, fontsize=6.8, rotation=30, ha="right")
ax[2].set_ylabel(r"mean Spearman $\rho$ (5-fold CV)")
ax[2].set_ylim(0.88, 0.985)
ax[2].set_title("(c) Three buckets are enough", fontsize=10)

# ---- (d) cross-scale ------------------------------------------------------
cs = R["cross_scale"]
labels = ["best single\nmethod", "router,\nthresholds\nfrom $n{=}100$",
          "router,\nthresholds\nfrom $n{=}900$"]
vv = [cs["n100_best"], cs["n100_router_native"], cs["n100_router_from900"]]
cc = ["0.75", "tab:orange", "crimson"]
ax[3].bar(range(3), vv, color=cc, edgecolor="k", lw=.5, width=.6)
for i, v in enumerate(vv):
    ax[3].text(i, v + .002, f"{v:.4f}", ha="center", fontsize=7.5)
ax[3].set_xticks(range(3))
ax[3].set_xticklabels(labels, fontsize=7)
ax[3].set_ylabel(r"mean Spearman $\rho$ at $n=100$")
ax[3].set_ylim(0.90, 0.975)
ax[3].set_title("(d) Thresholds transfer across a $9\\times$ change of scale",
                fontsize=10)

plt.tight_layout()
plt.savefig("fig3.pdf", bbox_inches="tight")
plt.savefig("fig3.png", bbox_inches="tight", dpi=150)
print("wrote fig3.pdf / fig3.png")

## What to take away

**One free statistic answers two questions.** $\bar P_{esc}$ says which
algorithm to run ($+0.031$ over the best single method, cross-validated,
transferring across scale) and how to damp the diffusions it selects among
($>98\%$ of a per-instance oracle, beating every constant $\alpha$).

**The selection result is the more durable of the two.** It does not depend on
any particular method staying state of the art: change the candidate pool,
refit three thresholds, and the mechanism stands.

**Two things are honestly open.** We have not verified novelty against the
algorithm-selection literature, and every graph here is synthetic — whether
$\bar P_{esc}$ predicts the regime on real networks is the most important
missing experiment.

## 13. Real-world validation

Everything above rests on generative topologies. This section applies the router
and rule-$\alpha$ to real networks **without refitting anything**.

Three genres, seven base graphs: citation (Cora, CiteSeer, PubMed + $k$-cores),
co-purchase (Amazon Computers, Photo), co-authorship (MS Academic CS, Physics).
Degree CV runs 0.77–1.97 and clustering 0.14–0.40 — neither reproduced by any
synthetic family here.

Data is fetched from GitHub-hosted mirrors (Planetoid; shchur/gnn-benchmark).

In [ ]:
import os, subprocess, sys
os.makedirs("real", exist_ok=True); os.chdir("real")
for u in ["https://github.com/kimiyoung/planetoid/raw/master/data/ind.cora.graph",
          "https://github.com/kimiyoung/planetoid/raw/master/data/ind.citeseer.graph",
          "https://github.com/kimiyoung/planetoid/raw/master/data/ind.pubmed.graph"]:
    subprocess.run(["curl","-sL","-o",u.split("/")[-1],u], check=True)
for f in ["amazon_electronics_computers","amazon_electronics_photo",
          "ms_academic_cs","ms_academic_phy"]:
    subprocess.run(["curl","-sL","-o",f+".npz",
        f"https://github.com/shchur/gnn-benchmark/raw/master/data/npz/{f}.npz"], check=True)
print("fetched:", sorted(os.listdir(".")))

### 13.1 The router on citation networks

In [ ]:
%%writefile real_experiment.py
"""
Real-world validation.

Everything here is applied WITHOUT REFITTING:
  * the 3-bucket router with thresholds fitted at n=900 on synthetic graphs
        Pesc_bar < 0.625        -> IRIE
        0.625 <= Pesc_bar < .875-> Monte Carlo, 200 samples
        Pesc_bar >= 0.875       -> Pesc alone
  * the damping rules alpha = clip(a + b*Pesc_bar, .4, 1) fitted at n=64

Topologies: Cora, CiteSeer and PubMed citation networks (Planetoid release) plus
their k-cores, which give real structure at a range of densities.  These have
genuine degree heterogeneity (CV 0.8-1.3, hubs 30-40x the mean) and real
clustering (0.14-0.38), neither of which any of the four synthetic families
reproduces.

Edge-probability models:
  uniform ranges (as in training) at four levels, plus WEIGHTED CASCADE
  p_ij = 1/max(deg_i, deg_j) -- the standard IC assignment, and genuinely
  out of distribution for our rules.

Usage:  python3 real_experiment.py build      # cache the topologies
        python3 real_experiment.py run <k>    # evaluate k pending instances
Results append to real_results.jsonl (resumable).
"""

import json
import os
import pickle
import sys
import time
import numpy as np
import networkx as nx

sys.path.insert(0, "..")
from escape_lib import (Instance, ground_truth, rho, timed, diffuse,
                        alpha_from_rule, m_pesc, m_sum_p, m_irie, m_mia,
                        m_mc, m_escape_pull)

CAL = json.load(open("../calibration.json"))
COEF = (CAL["intercept"], CAL["slope"])
AT = json.load(open("../alpha_transfer.json"))
PUSH_COEF = tuple(AT["rules"]["PPR push"])
ROUTER = [(0.0, 0.625, "IRIE"), (0.625, 0.875, "MC 200"),
          (0.875, 1.01, "P_esc alone")]

TOPO_FILE = "topologies.pkl"
CFG_FILE = "real_configs.json"
OUT = "real_results.jsonl"

PROB_MODELS = [
    ("uniform[0.02,0.08]", ("u", 0.02, 0.08)),
    ("uniform[0.10,0.25]", ("u", 0.10, 0.25)),
    ("uniform[0.25,0.45]", ("u", 0.25, 0.45)),
    ("uniform[0.45,0.75]", ("u", 0.45, 0.75)),
    ("weighted-cascade 1/deg", ("wc", 0, 0)),
]
PRIORS = ["dirichlet", "uniform", "sparse"]


def load_planetoid(name):
    g = pickle.load(open(f"ind.{name}.graph", "rb"), encoding="latin1")
    G = nx.Graph()
    for u, nb in g.items():
        for v in nb:
            if u != v:
                G.add_edge(int(u), int(v))
    G.remove_edges_from(nx.selfloop_edges(G))
    return G


def build():
    specs = [("cora", 1), ("cora", 2), ("cora", 3),
             ("citeseer", 1), ("citeseer", 2),
             ("pubmed", 4), ("pubmed", 5)]
    topos = {}
    for name, k in specs:
        G = load_planetoid(name)
        C = nx.k_core(G, k) if k > 1 else G
        C = nx.convert_node_labels_to_integers(C)
        lab = f"{name}-{k}core" if k > 1 else name
        deg = np.array([d for _, d in sorted(C.degree())], dtype=float)
        ei, ej = [], []
        for u, v in C.edges():
            a, b = (u, v) if u < v else (v, u)
            ei.append(a)
            ej.append(b)
        topos[lab] = dict(n=C.number_of_nodes(), ei=np.array(ei, dtype=np.int32),
                          ej=np.array(ej, dtype=np.int32), deg=deg,
                          clust=float(nx.average_clustering(C)),
                          deg_cv=float(deg.std() / deg.mean()))
        print(f"  {lab:16s} n={topos[lab]['n']:5d} m={len(ei):6d} "
              f"avg_deg={deg.mean():5.2f} cv={topos[lab]['deg_cv']:.2f} "
              f"clust={topos[lab]['clust']:.3f}")
    pickle.dump(topos, open(TOPO_FILE, "wb"))
    cfgs = []
    for lab in topos:
        for pi, (pname, _) in enumerate(PROB_MODELS):
            cfgs.append(dict(id=len(cfgs), topo=lab, prob=pname, pidx=pi,
                             prior=PRIORS[len(cfgs) % 3],
                             seed=1000 + len(cfgs)))
    json.dump(cfgs, open(CFG_FILE, "w"))
    print(f"\n{len(topos)} topologies, {len(cfgs)} instances queued")


def make_real_instance(topos, cfg):
    T = topos[cfg["topo"]]
    n, ei, ej, deg = T["n"], T["ei"], T["ej"], T["deg"]
    rng = np.random.default_rng(cfg["seed"])
    kind, lo, hi = PROB_MODELS[cfg["pidx"]][1]
    if kind == "u":
        p = rng.uniform(lo, hi, size=len(ei))
    else:                                    # weighted cascade, symmetrised
        p = 1.0 / np.maximum(deg[ei], deg[ej])
    p = np.clip(p, 1e-4, 0.999)
    if cfg["prior"] == "dirichlet":
        x = rng.dirichlet(np.full(n, 0.7))
    elif cfg["prior"] == "uniform":
        x = np.full(n, 1.0 / n)
    else:
        x = np.zeros(n)
        idx = rng.choice(n, size=max(1, n // 10), replace=False)
        x[idx] = rng.dirichlet(np.full(len(idx), 1.0))
    return Instance(n, ei, ej, p, x / x.sum())


def route(pe):
    for lo, hi, m in ROUTER:
        if lo <= pe < hi:
            return m
    return ROUTER[-1][2]


def run(count):
    topos = pickle.load(open(TOPO_FILE, "rb"))
    cfgs = json.load(open(CFG_FILE))
    done = set()
    if os.path.exists(OUT):
        for line in open(OUT):
            if line.strip():
                done.add(json.loads(line)["id"])
    todo = [c for c in cfgs if c["id"] not in done][:count]
    print(f"{len(done)} done, {len(cfgs)-len(done)} pending; running {len(todo)}")
    t0 = time.time()
    with open(OUT, "a") as fh:
        for cfg in todo:
            ins = make_real_instance(topos, cfg)
            gt = ground_truth(ins, 3000, seed=cfg["seed"] + 7)
            a_e = alpha_from_rule(ins, COEF)
            a_p = float(np.clip(PUSH_COEF[0] + PUSH_COEF[1] * ins.mean_pesc,
                                0.4, 1.0))
            METH = {
                "P_esc alone":     lambda i: m_pesc(i),
                "sum_j p_ij":      lambda i: m_sum_p(i),
                "IRIE":            lambda i: m_irie(i, 0.7),
                "push PPR rule-a": lambda i: diffuse(i, a_p, use_pesc=False,
                                                     orientation="push"),
                "escape rule-a":   lambda i: m_escape_pull(i, a_e),
                "MC 50":           lambda i: m_mc(i, 50),
                "MC 200":          lambda i: m_mc(i, 200),
            }
            if ins.n <= 4500:
                METH["MIA/PMIA"] = lambda i: m_mia(i, 1e-3)
            rec = dict(cfg)
            rec.update(n=ins.n, m=int(ins.m), mean_pesc=ins.mean_pesc,
                       alpha_rule=a_e, rho={}, ms={})
            for nm, fn in METH.items():
                s, dt = timed(fn, ins)
                rec["rho"][nm] = rho(s, gt)
                rec["ms"][nm] = dt * 1000
            rec["routed_to"] = route(ins.mean_pesc)
            rec["rho"]["ROUTER"] = rec["rho"][rec["routed_to"]]
            rec["ms"]["ROUTER"] = rec["ms"][rec["routed_to"]]
            fh.write(json.dumps(rec) + "\n")
            fh.flush()
            print(f"  [{cfg['id']:2d}] {cfg['topo']:16s} {cfg['prob']:22s} "
                  f"Pesc={ins.mean_pesc:.3f} -> {rec['routed_to']:12s} "
                  f"router={rec['rho']['ROUTER']:.3f} "
                  f"irie={rec['rho']['IRIE']:.3f}", flush=True)
    print(f"chunk done in {(time.time()-t0)/60:.1f} min")


if __name__ == "__main__":
    if sys.argv[1] == "build":
        build()
    else:
        run(int(sys.argv[2]))


In [ ]:
subprocess.run([sys.executable,"real_experiment.py","build"], check=True)
subprocess.run([sys.executable,"real_experiment.py","run","35"], check=True)

**Outcome: the routing premise fails here.** The router scores 0.9176 against
0.9637 for simply running MC 200 everywhere. Sampling wins *every* bucket
(0.968 / 0.992 / 0.929), and a router refit on real data degenerates to
selecting Monte Carlo throughout. The sparse regime in which IRIE led on
synthetic graphs has no counterpart on citation networks.

### 13.2 Scale, with the density regime held fixed

Snowball subgraphs at increasing $n$, with the edge probability tuned per
instance so $\bar P_{esc}$ hits a fixed target — the only way to attribute a
trend to scale rather than to regime.

In [ ]:
%%writefile scale_real.py
"""
Does the sampling advantage survive scale?

Hypothesis.  A Monte-Carlo estimate of y* has per-node error O(1/sqrt(W)),
independent of n.  But as n grows the y* values crowd together, so a FIXED
sample budget corrupts an increasing fraction of ranking pairs.  A deterministic
diffusion has no such term.  If that is right, MC's lead should shrink with n
while the diffusion's accuracy holds, and there is some n beyond which the
deterministic score wins.

Design.  BFS-sampled induced subgraphs of one real network at increasing n, with
the edge probability tuned per instance so that Pesc_bar hits a FIXED target.
That holds the density regime constant and varies only n, which is the only way
to attribute any trend to scale rather than to regime.

Usage:  python3 scale_real.py <genre> <target_pesc> <n1,n2,...>
Appends to scale_real.jsonl.
"""

import json
import sys
import time
import numpy as np
import scipy.sparse as sp
import networkx as nx

sys.path.insert(0, "..")
from escape_lib import (Instance, ground_truth, rho, timed, diffuse,
                        alpha_from_rule, m_pesc, m_irie, m_mc, m_escape_pull)

CAL = json.load(open("../calibration.json"))
COEF = (CAL["intercept"], CAL["slope"])
AT = json.load(open("../alpha_transfer.json"))
PUSH_COEF = tuple(AT["rules"]["PPR push"])
OUT = "scale_real.jsonl"

FILES = {"coauthor-physics": "ms_academic_phy.npz",
         "coauthor-cs": "ms_academic_cs.npz",
         "amazon-computers": "amazon_electronics_computers.npz",
         "amazon-photo": "amazon_electronics_photo.npz"}


def load_graph(genre):
    d = np.load(FILES[genre], allow_pickle=True)
    A = sp.csr_matrix((d["adj_data"], d["adj_indices"], d["adj_indptr"]),
                      shape=tuple(d["adj_shape"]))
    A = ((A + A.T) > 0).astype(np.int8)
    A.setdiag(0)
    A.eliminate_zeros()
    return nx.from_scipy_sparse_array(A)


def bfs_ball(G, n_target, seed):
    """Snowball sample: preserves local density far better than a random
    node sample, which would artificially sparsify the subgraph."""
    rng = np.random.default_rng(seed)
    nodes = list(G.nodes())
    start = int(nodes[rng.integers(len(nodes))])
    seen = {start}
    frontier = [start]
    while len(seen) < n_target and frontier:
        nxt = []
        for u in frontier:
            for v in G.neighbors(u):
                if v not in seen:
                    seen.add(v)
                    nxt.append(v)
                    if len(seen) >= n_target:
                        break
            if len(seen) >= n_target:
                break
        frontier = nxt
    return nx.convert_node_labels_to_integers(G.subgraph(seen).copy())


def edges_of(G):
    ei, ej = [], []
    for u, v in G.edges():
        a, b = (u, v) if u < v else (v, u)
        ei.append(a)
        ej.append(b)
    return np.array(ei, dtype=np.int32), np.array(ej, dtype=np.int32)


def tune_p(n, ei, ej, target, x):
    """Binary-search a constant edge probability giving mean Pesc == target,
    so every size sits in the same density regime."""
    lo, hi = 1e-5, 0.999
    for _ in range(40):
        mid = (lo + hi) / 2
        inst = Instance(n, ei, ej, np.full(len(ei), mid), x)
        if inst.mean_pesc < target:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


def run(genre, target, sizes):
    G = load_graph(genre)
    print(f"{genre}: n={G.number_of_nodes()} m={G.number_of_edges()}")
    with open(OUT, "a") as fh:
        for n_t in sizes:
            t0 = time.time()
            S = bfs_ball(G, n_t, seed=11)
            n = S.number_of_nodes()
            ei, ej = edges_of(S)
            rng = np.random.default_rng(5)
            x = rng.dirichlet(np.full(n, 0.7))
            x /= x.sum()
            p = tune_p(n, ei, ej, target, x)
            inst = Instance(n, ei, ej, np.full(len(ei), p), x)

            gtA = ground_truth(inst, 3000, seed=101)
            gtB = ground_truth(inst, 3000, seed=202)   # GT reliability check
            gt_self = rho(gtA, gtB)

            a_e = alpha_from_rule(inst, COEF)
            a_p = float(np.clip(PUSH_COEF[0] + PUSH_COEF[1] * inst.mean_pesc,
                                0.4, 1.0))
            METH = {
                "P_esc alone":   lambda i: m_pesc(i),
                "IRIE":          lambda i: m_irie(i, 0.7),
                "escape rule-a": lambda i: m_escape_pull(i, a_e),
                "push PPR rule-a": lambda i: diffuse(i, a_p, use_pesc=False,
                                                     orientation="push"),
                "MC 50":         lambda i: m_mc(i, 50),
                "MC 200":        lambda i: m_mc(i, 200),
            }
            rec = dict(genre=genre, target=target, n=n, m=int(inst.m),
                       p=float(p), mean_pesc=float(inst.mean_pesc),
                       gt_self_consistency=float(gt_self), rho={}, ms={})
            for nm, fn in METH.items():
                s, dt = timed(fn, inst)
                rec["rho"][nm] = rho(s, gtA)
                rec["ms"][nm] = dt * 1000
            fh.write(json.dumps(rec) + "\n")
            fh.flush()
            print(f"  n={n:6d} m={inst.m:7d} p={p:.4f} Pesc={inst.mean_pesc:.3f} "
                  f"| GT self-consistency {gt_self:.4f} | "
                  f"MC200 {rec['rho']['MC 200']:.3f} "
                  f"escape {rec['rho']['escape rule-a']:.3f} "
                  f"IRIE {rec['rho']['IRIE']:.3f} "
                  f"({time.time()-t0:.0f}s)", flush=True)


if __name__ == "__main__":
    run(sys.argv[1], float(sys.argv[2]),
        [int(z) for z in sys.argv[3].split(",")])


In [ ]:
for tgt, sizes in [(0.6,"2000,4000,8000,16000,32000"), (0.9,"2000,8000,32000"),
                   (0.97,"8000,32000")]:
    subprocess.run([sys.executable,"scale_real.py","coauthor-physics",str(tgt),sizes], check=True)

### 13.3 The dense regime: where the free score wins

30 instances, four non-citation genres, five sizes, two snowball seeds, density
regime held at $\bar P_{esc}=0.90$.

In [ ]:
%%writefile dense_panel.py
"""
Where does the free score beat sampling, and is there a crossover in n?

Earlier evidence pulls in two directions.  On citation k-cores at n=1257-4201 in
the dense bucket, 200-sample Monte Carlo beat Pesc alone.  On coauthor-physics
at n=8000-32000 in the dense regime, Pesc alone beat it.  If both are right,
there is a crossover in n -- which would matter, because it is exactly the
regime the router dispatches to.

Design: four genres x five sizes x two BFS seeds, edge probability tuned per
instance so Pesc_bar hits a fixed dense target, so only n and genre vary.
Enough instances for a paired test rather than an anecdote.

Usage: python3 dense_panel.py <count>     (resumable, appends to dense_panel.jsonl)
"""

import json
import os
import sys
import time
import numpy as np

sys.path.insert(0, "..")
from escape_lib import (Instance, ground_truth, rho, timed, m_pesc, m_sum_p,
                        m_irie, m_mc, m_escape_pull, alpha_from_rule)
from scale_real import load_graph, bfs_ball, edges_of, COEF

OUT = "dense_panel.jsonl"
GENRES = ["coauthor-physics", "coauthor-cs", "amazon-computers", "amazon-photo"]
SIZES = [1000, 2000, 4000, 8000, 16000]
SEEDS = [11, 23]
TARGET = 0.90

CFGS = [dict(id=i, genre=g, n_t=s, seed=sd)
        for i, (g, s, sd) in enumerate(
            (g, s, sd) for g in GENRES for s in SIZES for sd in SEEDS)]


def tune(n, ei, ej, x, target, seed):
    rng = np.random.default_rng(seed)
    base = rng.uniform(0.5, 1.5, len(ei))       # mild p-heterogeneity
    lo, hi = 1e-6, 1.0
    for _ in range(40):
        mid = (lo + hi) / 2
        p = np.clip(base * mid, 1e-6, 0.999)
        if Instance(n, ei, ej, p, x).mean_pesc < target:
            lo = mid
        else:
            hi = mid
    return np.clip(base * (lo + hi) / 2, 1e-6, 0.999)


def main(count):
    done = set()
    if os.path.exists(OUT):
        for line in open(OUT):
            if line.strip():
                done.add(json.loads(line)["id"])
    todo = [c for c in CFGS if c["id"] not in done][:count]
    print(f"{len(done)} done, {len(CFGS)-len(done)} pending; running {len(todo)}")
    cache = {}
    with open(OUT, "a") as fh:
        for cfg in todo:
            t0 = time.time()
            if cfg["genre"] not in cache:
                cache.clear()
                cache[cfg["genre"]] = load_graph(cfg["genre"])
            G = cache[cfg["genre"]]
            if G.number_of_nodes() < cfg["n_t"]:
                continue
            S = bfs_ball(G, cfg["n_t"], seed=cfg["seed"])
            n = S.number_of_nodes()
            ei, ej = edges_of(S)
            rng = np.random.default_rng(cfg["seed"] + 5)
            x = rng.dirichlet(np.full(n, 0.7))
            x /= x.sum()
            p = tune(n, ei, ej, x, TARGET, cfg["seed"])
            inst = Instance(n, ei, ej, p, x)
            gt = ground_truth(inst, 3000, seed=cfg["seed"] * 7 + 1)
            gt2 = ground_truth(inst, 3000, seed=cfg["seed"] * 13 + 3)
            a_e = alpha_from_rule(inst, COEF)
            METH = {"P_esc alone": lambda i: m_pesc(i),
                    "sum_j p_ij": lambda i: m_sum_p(i),
                    "IRIE": lambda i: m_irie(i, 0.7),
                    "escape rule-a": lambda i: m_escape_pull(i, a_e),
                    "MC 50": lambda i: m_mc(i, 50),
                    "MC 200": lambda i: m_mc(i, 200),
                    "MC 800": lambda i: m_mc(i, 800)}
            rec = dict(cfg)
            rec.update(n=n, m=int(inst.m), mean_pesc=float(inst.mean_pesc),
                       gt_self=float(rho(gt, gt2)), rho={}, ms={})
            for nm, fn in METH.items():
                s, dt = timed(fn, inst)
                rec["rho"][nm] = rho(s, gt)
                rec["ms"][nm] = dt * 1000
            fh.write(json.dumps(rec) + "\n")
            fh.flush()
            print(f"  [{cfg['id']:2d}] {cfg['genre']:17s} n={n:6d} m={inst.m:7d} "
                  f"Pesc={inst.mean_pesc:.3f} | P_esc {rec['rho']['P_esc alone']:.4f} "
                  f"MC200 {rec['rho']['MC 200']:.4f} "
                  f"diff {rec['rho']['P_esc alone']-rec['rho']['MC 200']:+.4f} "
                  f"({time.time()-t0:.0f}s)", flush=True)


if __name__ == "__main__":
    main(int(sys.argv[1]))


In [ ]:
for _ in range(4):
    subprocess.run([sys.executable,"dense_panel.py","12"], check=True)

In [ ]:
import json, numpy as np
from scipy.stats import wilcoxon
D=[json.loads(l) for l in open("dense_panel.jsonl") if l.strip() and json.loads(l)["n"]>100]
M=["P_esc alone","sum_j p_ij","IRIE","escape rule-a","MC 50","MC 200","MC 800"]
print(f"{len(D)} valid instances (degenerate n=1 snowball samples dropped)")
print(f"GT self-consistency: min {min(r['gt_self'] for r in D):.4f}\n")
print(f"{'method':16s} {'mean rho':>9s} {'mean ms':>9s}")
for m in sorted(M, key=lambda k:-np.mean([r['rho'][k] for r in D])):
    print(f"  {m:14s} {np.mean([r['rho'][m] for r in D]):9.4f} "
          f"{np.mean([r['ms'][m] for r in D]):9.1f}")
pe=np.array([r["rho"]["P_esc alone"] for r in D])
print()
for m in ["MC 800","MC 200","MC 50","IRIE","escape rule-a"]:
    d=pe-np.array([r["rho"][m] for r in D]); nz=d[np.abs(d)>1e-12]
    print(f"  P_esc vs {m:14s} {d.mean():+.4f}  wins {(d>0).sum():2d}/{len(d)}  "
          f"p={wilcoxon(nz).pvalue:.2e}")
print("\nby genre:")
for g in sorted(set(r["genre"] for r in D)):
    s=[r for r in D if r["genre"]==g]
    d=np.array([r["rho"]["P_esc alone"]-r["rho"]["MC 200"] for r in s])
    print(f"  {g:18s} k={len(s):2d}  diff {d.mean():+.4f}  wins {(d>0).sum()}/{len(d)}")

In [ ]:
%%writefile make_figure4.py
"""Figure 4: real-world validation."""
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REAL = [json.loads(l) for l in open("real_results.jsonl") if l.strip()]
DENSE = [json.loads(l) for l in open("dense_panel.jsonl")
         if l.strip() and json.loads(l)["n"] > 100]
SCALE = [json.loads(l) for l in open("scale_real.jsonl") if l.strip()]

plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": .3,
                     "figure.dpi": 160})
fig, axes = plt.subplots(2, 2, figsize=(11.0, 7.4))
ax = axes.ravel()

# ---- (a) citation networks: the router's buckets vs reality ---------------
E = [0, .625, .875, 1.01]
NAMES = ["$<0.625$\n(routed: IRIE)", "$0.625$–$0.875$\n(routed: MC 200)",
         "$\\geq 0.875$\n(routed: $P_{esc}$)"]
M = ["P_esc alone", "IRIE", "escape rule-a", "MC 200"]
COL = {"P_esc alone": "0.35", "IRIE": "tab:green",
       "escape rule-a": "tab:orange", "MC 200": "tab:blue"}
LB = {"P_esc alone": r"$P_{esc}$ alone", "IRIE": "IRIE",
      "escape rule-a": "escape rule-$\\alpha$", "MC 200": "MC 200"}
w = 0.2
for k, m in enumerate(M):
    ys = []
    for b in range(3):
        sel = [r for r in REAL if E[b] <= r["mean_pesc"] < E[b + 1]]
        ys.append(np.mean([r["rho"][m] for r in sel]) if sel else 0)
    ax[0].bar(np.arange(3) + (k - 1.5) * w, ys, w, color=COL[m],
              edgecolor="k", lw=.4, label=LB[m])
ax[0].set_xticks(range(3))
ax[0].set_xticklabels(NAMES, fontsize=7)
ax[0].set_ylabel(r"mean Spearman $\rho$")
ax[0].set_ylim(0, 1.08)
ax[0].set_title("(a) Citation networks: sampling wins every bucket",
                fontsize=10)
ax[0].legend(fontsize=6.6, ncol=2, loc="lower left", framealpha=.93)

# ---- (b) dense regime, by genre -------------------------------------------
genres = ["amazon-photo", "amazon-computers", "coauthor-cs",
          "coauthor-physics"]
pe = [np.mean([r["rho"]["P_esc alone"] for r in DENSE if r["genre"] == g])
      for g in genres]
mc = [np.mean([r["rho"]["MC 200"] for r in DENSE if r["genre"] == g])
      for g in genres]
mc8 = [np.mean([r["rho"]["MC 800"] for r in DENSE if r["genre"] == g])
       for g in genres]
xp = np.arange(len(genres))
ax[1].bar(xp - .25, pe, .25, color="0.35", edgecolor="k", lw=.4,
          label=r"$P_{esc}$ alone (free)")
ax[1].bar(xp, mc, .25, color="tab:blue", edgecolor="k", lw=.4,
          label="MC 200 (263 ms)")
ax[1].bar(xp + .25, mc8, .25, color="tab:cyan", edgecolor="k", lw=.4,
          label="MC 800 (1035 ms)")
ax[1].set_xticks(xp)
ax[1].set_xticklabels([g.replace("-", "\n") for g in genres], fontsize=6.8)
ax[1].set_ylim(0.93, 1.005)
ax[1].set_ylabel(r"mean Spearman $\rho$")
ax[1].set_title(r"(b) Dense regime ($\bar{P}_{esc}=0.90$), 30 instances",
                fontsize=10)
ax[1].legend(fontsize=6.8, loc="lower left", framealpha=.93)

# ---- (c) paired differences in the dense regime ---------------------------
d200 = np.array([r["rho"]["P_esc alone"] - r["rho"]["MC 200"] for r in DENSE])
d800 = np.array([r["rho"]["P_esc alone"] - r["rho"]["MC 800"] for r in DENSE])
d50 = np.array([r["rho"]["P_esc alone"] - r["rho"]["MC 50"] for r in DENSE])
bp = ax[2].boxplot([d50, d200, d800], vert=False, widths=.6,
                   showfliers=False, patch_artist=True,
                   medianprops=dict(color="k", lw=1.4))
for patch, c in zip(bp["boxes"], ["tab:cyan", "tab:blue", "navy"]):
    patch.set_facecolor(c)
    patch.set_alpha(.55)
for i, d in enumerate([d50, d200, d800]):
    ax[2].plot([d.mean()], [i + 1], "D", color="k", ms=5, zorder=5)
ax[2].axvline(0, color="k", lw=1.2, ls="--")
ax[2].set_yticklabels(["vs MC 50\n$p=4\\times10^{-9}$",
                       "vs MC 200\n$p=4\\times10^{-4}$",
                       "vs MC 800\n$p=0.32$"], fontsize=7)
ax[2].set_xlabel(r"paired difference in $\rho$ ($P_{esc}$ alone minus sampling)")
ax[2].set_title("(c) The free score beats 200-sample MC", fontsize=10)

# ---- (d) scale sweep, regime held fixed -----------------------------------
for tgt, c, mk in [(0.6, "tab:green", "o"), (0.9, "tab:red", "s")]:
    sel = sorted([r for r in SCALE if r["target"] == tgt],
                 key=lambda z: z["n"])
    if not sel:
        continue
    ns = [r["n"] for r in sel]
    ax[3].plot(ns, [r["rho"]["MC 200"] for r in sel], "-" + mk, color=c,
               ms=5, lw=1.6, label=f"MC 200, $\\bar{{P}}_{{esc}}={tgt}$")
    ax[3].plot(ns, [r["rho"]["P_esc alone"] for r in sel], "--" + mk,
               color=c, ms=5, lw=1.6, alpha=.6, mfc="none",
               label=f"$P_{{esc}}$ alone, $\\bar{{P}}_{{esc}}={tgt}$")
ax[3].set_xscale("log")
ax[3].set_xlabel("$n$ (log scale), coauthor-physics subgraphs")
ax[3].set_ylabel(r"Spearman $\rho$")
ax[3].set_title("(d) Scale, with the density regime held fixed", fontsize=10)
ax[3].legend(fontsize=6.4, loc="lower left", framealpha=.93)

plt.tight_layout()
plt.savefig("../fig4.pdf", bbox_inches="tight")
plt.savefig("../fig4.png", bbox_inches="tight", dpi=150)
print("wrote fig4.pdf / fig4.png")


In [ ]:
exec(open("make_figure4.py").read().split('"""',2)[-1])
os.chdir("..")

### What real data says

**The router does not transfer.** On citation networks sampling wins every
bucket; a refit router just picks it everywhere. The premise that no method
wins everywhere is a property of the synthetic families, not a general fact.
Section 9's conclusions should not be extrapolated.

**The dense bucket does transfer, and decisively.** Across 30 co-purchase and
co-authorship instances, $P_{esc}$ used alone — no diffusion, computed at
initialisation — outranks a 200-sample Monte Carlo on 24/30
($p=4.2\times10^{-4}$) at roughly $10^4$ times less cost. Only 800 samples
overtake it, insignificantly, at 1035 ms against 0.03 ms. The margin tracks
degree heterogeneity: $+0.027$/$+0.029$ on Amazon, $+0.004$/$+0.002$ on
co-authorship.

It is **not** a scale effect (present already at $n=10^3$, does not widen) and
**not** an artefact of homogeneous edge probabilities (holds at CV $=0$, $0.29$,
$0.55$ in $p$). When most vertices are connected, $y^*_i \approx P_{esc}(i)
\times$ (giant-component mass), so the free statistic is nearly the analytic
answer while a few hundred samples still carry visible noise.

**The $\alpha$ rule transfers, for the wrong reason.** It still beats every
constant $\alpha$ (91% and 89% of oracle) but the correlation that motivated it
collapses ($r=+0.965 \to +0.225$). It picks good damping largely because it
picks high damping, and high is right here.

## 14. Message passing: the proper deterministic competitor

$P_{esc}$ is not a mechanism of its own. Karrer, Newman & Zdeborová
(*Phys. Rev. Lett.* 113:208702, 2014) compute per-node giant-cluster membership
by message passing:

$$\nu_{i\to j}=1-\prod_{k\in N(i)\setminus j}(1-p_{ik}\nu_{k\to i}),
\qquad \mu_i=1-\prod_{k\in N(i)}(1-p_{ik}\nu_{k\to i}).$$

Initialise $\nu\equiv1$, evaluate $\mu$ once with no updates, and you get
$1-\prod_k(1-p_{ik})=P_{esc}(i)$ **exactly**. The escape probability is the
zeroth iterate. The honest deterministic competitor is therefore the recursion
run properly — so we run it.

In [ ]:
%%writefile mp_percolation.py
"""
Message-passing percolation (Karrer, Newman & Zdeborova, PRL 113:208702, 2014).

For each directed edge i->j let nu_{i->j} be the probability that i connects to
the giant cluster through some neighbour other than j:

    nu_{i->j} = 1 - prod_{k in N(i)\\j} (1 - p_ik * nu_{k->i})

and the probability that vertex i belongs to the giant cluster is

    mu_i      = 1 - prod_{k in N(i)}   (1 - p_ik * nu_{k->i}).

Initialising nu == 1 and evaluating mu once, with no message updates, gives

    mu_i = 1 - prod_k (1 - p_ik) = Pesc(i),

so the escape probability is exactly the correlation-free (eta = 1) limit of
this recursion -- its zeroth iterate.  That identity is why Pesc is treated in
this work as a free proxy for message passing rather than as a mechanism of its
own.

Products are evaluated in log space; the excluded-neighbour term is removed by
subtraction rather than division, which is stable when p*nu approaches 1.
"""

import numpy as np


def _directed(inst):
    """Two directed edges per undirected edge; rev[2t]=2t+1, rev[2t+1]=2t."""
    ei, ej, p = inst.ei, inst.ej, inst.p
    src = np.empty(2 * len(ei), dtype=np.int64)
    dst = np.empty(2 * len(ei), dtype=np.int64)
    pe = np.empty(2 * len(ei), dtype=float)
    src[0::2], dst[0::2] = ei, ej
    src[1::2], dst[1::2] = ej, ei
    pe[0::2] = p
    pe[1::2] = p
    rev = np.arange(2 * len(ei)) ^ 1
    return src, dst, pe, rev


def mp_percolation(inst, n_iter=200, tol=1e-10, damping=0.0, return_iters=False):
    """Per-node probability of belonging to the giant cluster."""
    n = inst.n
    src, dst, pe, rev = _directed(inst)
    if len(pe) == 0:
        out = np.zeros(n)
        return (out, 0) if return_iters else out
    nu = np.ones(len(pe))
    EPS = 1e-15
    it = 0
    for it in range(1, n_iter + 1):
        # contribution of edge (k->i) to node i = log(1 - p * nu_{k->i})
        contrib = np.log(np.clip(1.0 - pe * nu, EPS, 1.0))
        # L[i] = sum over all incoming edges k->i
        L = np.bincount(dst, weights=contrib, minlength=n)
        # for edge e = i->j : exclude the term arriving on rev[e] = j->i
        nu_new = 1.0 - np.exp(L[src] - contrib[rev])
        nu_new = np.clip(nu_new, 0.0, 1.0)
        if damping > 0:
            nu_new = damping * nu + (1 - damping) * nu_new
        delta = np.abs(nu_new - nu).max()
        nu = nu_new
        if delta < tol:
            break
    contrib = np.log(np.clip(1.0 - pe * nu, EPS, 1.0))
    L = np.bincount(dst, weights=contrib, minlength=n)
    mu = 1.0 - np.exp(L)
    return (mu, it) if return_iters else mu


def m_mp(inst, n_iter=200, tol=1e-10):
    """Ranking score: giant-cluster membership probability."""
    return mp_percolation(inst, n_iter=n_iter, tol=tol)


def m_mp_weighted(inst, n_iter=200, tol=1e-10):
    """Membership probability scaled by the prior mass it would reach.

    y*_i ~= mu_i * (prior mass of the giant cluster) + (local mass if not in it).
    The first term is a constant across i, so for RANKING mu_i alone is the
    relevant quantity; this variant adds the second-order local term.
    """
    mu = mp_percolation(inst, n_iter=n_iter, tol=tol)
    giant_mass = float(np.dot(mu, inst.x))
    return mu * giant_mass + (1.0 - mu) * inst.x


def m_mp_k(inst, k):
    """Message passing truncated at k iterations.  k=0 is exactly Pesc."""
    n = inst.n
    src, dst, pe, rev = _directed(inst)
    if len(pe) == 0:
        return np.zeros(n)
    nu = np.ones(len(pe))
    EPS = 1e-15
    for _ in range(k):
        contrib = np.log(np.clip(1.0 - pe * nu, EPS, 1.0))
        L = np.bincount(dst, weights=contrib, minlength=n)
        nu = np.clip(1.0 - np.exp(L[src] - contrib[rev]), 0.0, 1.0)
    contrib = np.log(np.clip(1.0 - pe * nu, EPS, 1.0))
    return 1.0 - np.exp(np.bincount(dst, weights=contrib, minlength=n))


In [ ]:
from mp_percolation import mp_percolation, m_mp_k
import numpy as np
ins = make_instance("er", 400, 0.10, 0.30, 3)
print("k=0 truncation equals P_esc exactly:",
      np.allclose(m_mp_k(ins, 0), ins.pesc, atol=1e-12))
mu, it = mp_percolation(ins, return_iters=True)
print(f"converged in {it} iterations; mu in [{mu.min():.3f}, {mu.max():.3f}]")

In [ ]:
%%writefile real/mp_compare.py
"""
The decisive comparison: message passing versus its own eta=1 limit.

Re-runs the 30 real dense-regime instances and adds
  * full message passing (iterated to convergence),
  * message passing truncated at k = 0,1,2,3 iterations (k=0 IS Pesc),
so we can see how much of the ranking signal the free zeroth iterate captures
and how quickly the remainder is recovered.
"""

import json
import sys
import time
import numpy as np
from scipy.stats import wilcoxon

sys.path.insert(0, "..")
from escape_lib import Instance, ground_truth, rho, timed, m_pesc, m_mc
from mp_percolation import mp_percolation, m_mp_k
from scale_real import load_graph, bfs_ball, edges_of
from dense_panel import tune, TARGET

CFGS = [json.loads(l) for l in open("dense_panel.jsonl") if l.strip()]
CFGS = [c for c in CFGS if c["n"] > 100]
print(f"re-running {len(CFGS)} dense instances with message passing")

rows = []
cache = {}
t0 = time.time()
for c in CFGS:
    if c["genre"] not in cache:
        cache.clear()
        cache[c["genre"]] = load_graph(c["genre"])
    S = bfs_ball(cache[c["genre"]], c["n_t"], seed=c["seed"])
    n = S.number_of_nodes()
    ei, ej = edges_of(S)
    rng = np.random.default_rng(c["seed"] + 5)
    x = rng.dirichlet(np.full(n, 0.7))
    x /= x.sum()
    p = tune(n, ei, ej, x, TARGET, c["seed"])
    inst = Instance(n, ei, ej, p, x)
    gt = ground_truth(inst, 3000, seed=c["seed"] * 7 + 1)

    r = dict(genre=c["genre"], n=n, m=int(inst.m), mean_pesc=float(inst.mean_pesc),
             rho={}, ms={})
    s, dt = timed(m_pesc, inst)
    r["rho"]["Pesc (k=0)"] = rho(s, gt)
    r["ms"]["Pesc (k=0)"] = dt * 1000
    for k in (1, 2, 3):
        s, dt = timed(m_mp_k, inst, k=k)
        r["rho"][f"MP k={k}"] = rho(s, gt)
        r["ms"][f"MP k={k}"] = dt * 1000
    t1 = time.perf_counter()
    mu, iters = mp_percolation(inst, return_iters=True)
    r["rho"]["MP full"] = rho(mu, gt)
    r["ms"]["MP full"] = (time.perf_counter() - t1) * 1000
    r["mp_iters"] = int(iters)
    s, dt = timed(m_mc, inst, n_samples=200)
    r["rho"]["MC 200"] = rho(s, gt)
    r["ms"]["MC 200"] = dt * 1000
    rows.append(r)
    print(f"  {c['genre']:17s} n={n:6d} | Pesc {r['rho']['Pesc (k=0)']:.4f} "
          f"k1 {r['rho']['MP k=1']:.4f} k2 {r['rho']['MP k=2']:.4f} "
          f"full {r['rho']['MP full']:.4f} ({iters} it) "
          f"MC200 {r['rho']['MC 200']:.4f}", flush=True)

json.dump(rows, open("mp_dense.json", "w"), indent=1)

M = ["Pesc (k=0)", "MP k=1", "MP k=2", "MP k=3", "MP full", "MC 200"]
print("\n" + "=" * 66)
print(f"{'method':14s} {'mean rho':>9s} {'mean ms':>9s} {'vs Pesc':>10s} {'p':>10s}")
pe = np.array([r["rho"]["Pesc (k=0)"] for r in rows])
for m in M:
    v = np.array([r["rho"][m] for r in rows])
    t = np.mean([r["ms"][m] for r in rows])
    if m == "Pesc (k=0)":
        print(f"{m:14s} {v.mean():9.4f} {t:9.2f} {'--':>10s} {'--':>10s}")
        continue
    d = v - pe
    nz = d[np.abs(d) > 1e-12]
    p = wilcoxon(nz).pvalue if len(nz) else 1.0
    print(f"{m:14s} {v.mean():9.4f} {t:9.2f} {d.mean():+10.4f} {p:10.2e}")
print(f"\nmean MP iterations to convergence: {np.mean([r['mp_iters'] for r in rows]):.1f}")
mp = np.array([r["rho"]["MP full"] for r in rows])
mc = np.array([r["rho"]["MC 200"] for r in rows])
d = mp - mc
print(f"MP full vs MC 200: {d.mean():+.4f}, MP wins {(d>0).sum()}/{len(d)}, "
      f"p={wilcoxon(d[np.abs(d)>1e-12]).pvalue:.2e}")
print(f"cost ratio MC200 / MPfull : "
      f"{np.mean([r['ms']['MC 200'] for r in rows])/np.mean([r['ms']['MP full'] for r in rows]):.1f}x")
print(f"cost ratio MPfull / Pesc  : "
      f"{np.mean([r['ms']['MP full'] for r in rows])/max(np.mean([r['ms']['Pesc (k=0)'] for r in rows]),1e-6):.0f}x")
print(f"\ntotal runtime {(time.time()-t0)/60:.1f} min")


In [ ]:
import os, subprocess, sys
os.chdir("real")
subprocess.run(["cp","../mp_percolation.py","."], check=True)
subprocess.run([sys.executable,"mp_compare.py"], check=True)
os.chdir("..")

### What message passing settles

| method | mean rho | ms |
|---|---|---|
| $P_{esc}$, i.e. $k=0$ | 0.9852 | ~0 |
| message passing, $k=1$ | 0.9919 | 6.0 |
| message passing, $k=2$ | 0.9928 | 3.2 |
| **message passing, converged** | **0.9931** | 38.0 |
| Monte Carlo, 200 samples | 0.9741 | 266.2 |

**Message passing beats its own $\eta=1$ limit** by $+0.0079$
($p=1.9\times10^{-9}$). Any claim that $P_{esc}$ is the right deterministic
score for this regime is wrong, and the paper does not make it.

**The deterministic family beats sampling**: converged MP outranks a 200-sample
Monte Carlo by $+0.0191$, winning 29/30 ($p=9.3\times10^{-9}$), at a seventh of
the cost. This confirms a point Newman (*Proc. R. Soc. A* 479:20220774, 2023)
makes directly — simulation gives an approximate answer subject to sampling
error where message passing gives the value. We record it as confirmation, not
as a finding.

**The ranking signal saturates almost immediately.** Two iterations reach within
$0.0003$ of convergence at a twelfth of the cost; the free zeroth iterate is
within $0.008$. For ranking in the dense regime, the correlations that message
passing corrects are largely rank-preserving. That is the one part of this
section we offer as our own.

## 15. A modern sampling baseline: reverse-reachable sets

The modern sampling machinery for influence is the reverse-reachable-set
framework (Borgs et al. SODA'14; IMM SIGMOD'15), whose current generation —
SUBSIM/HIST (SIGMOD'20, TODS'22) — is the state of the art with guarantees.
A comparative study should include it.

**Estimator.** Sample a root $r\sim x$, sample a world lazily; on an
undirected support $\mathrm{RR}(r)=C(r)$, and
$\mathbb{E}[\mathbf 1\{i\in\mathrm{RR}\}]=\sum_r x_r P(i\sim r)=y^*_i$
**exactly** — the singleton-spread special case, root drawn from the prior.
Sets are generated lazily and level-synchronously (only edges incident to the
growing set are examined, each undirected edge at most once, Bernoulli draws
vectorised over the frontier — the subset-sampling idea in array form).

In [ ]:
%%writefile rr_ranking.py
"""
Reverse-reachable-set ranking estimator (the RIS framework).

Estimator.  Sample a root r ~ x; sample a possible world lazily; the
reverse-reachable set is RR(r) = C(r), the connected component of r.  Then

    E[ 1{ i in RR } ]  =  sum_r x_r P(i ~ r)  =  y*_i        (exactly),

so the fraction of theta sets containing i is an unbiased estimate of the
expected reachable prior mass.  This is the singleton-spread special case of
reverse-reachable-set sampling (Borgs et al., SODA 2014; Tang et al., SIGMOD
2015), with the root drawn from the node-weight prior as in weighted extensions.

Construction.  Sets are generated lazily and level-synchronously: only edges
incident to the growing set are ever examined, each undirected edge at most
once, with the per-edge Bernoulli draws vectorised over the whole frontier --
the subset-sampling idea behind modern RR-set generators (Guo et al., SIGMOD
2020 / TODS 2022) in array form.  A visited-stamp array avoids O(n) resets, so
per-set cost is proportional to the edges incident to the realised set, not to
the graph.
"""

import numpy as np


def m_rr(inst, n_sets=200, seed=999):
    """Ranking score: per-node containment frequency over n_sets RR sets."""
    rng = np.random.default_rng(seed)
    indptr, indices, probs = inst.P.indptr, inst.P.indices, inst.P.data
    n = inst.n
    hits = np.zeros(n)
    stamp = np.full(n, -1, dtype=np.int64)
    roots = rng.choice(n, size=n_sets, p=inst.x)
    for t in range(n_sets):
        root = roots[t]
        stamp[root] = t
        reached = [np.array([root])]
        frontier = np.array([root])
        while frontier.size:
            starts = indptr[frontier]
            counts = indptr[frontier + 1] - starts
            total = int(counts.sum())
            if total == 0:
                break
            # flat gather of all adjacency slices of the frontier
            offs = np.repeat(np.cumsum(counts) - counts, counts)
            flat = np.repeat(starts, counts) + (np.arange(total) - offs)
            targets = indices[flat]
            fresh = stamp[targets] != t
            if not fresh.any():
                break
            targets = targets[fresh]
            live = rng.random(targets.size) < probs[flat[fresh]]
            if not live.any():
                break
            new = np.unique(targets[live])
            new = new[stamp[new] != t]          # dedupe across frontier nodes
            if new.size == 0:
                break
            stamp[new] = t
            reached.append(new)
            frontier = new
        hits[np.concatenate(reached)] += 1.0
    return hits / n_sets


In [ ]:
import numpy as np
from escape_lib import make_instance, ground_truth, rho
from rr_ranking import m_rr

# correctness: high-theta RR must converge to ground truth
ins = make_instance("er", 80, 0.10, 0.30, 7)
gt  = ground_truth(ins, 60000, seed=3)
est = m_rr(ins, n_sets=60000, seed=4)
print("max |RR - y*|    :", float(np.abs(est - gt).max()),
      "  (MC noise ~", round(1/np.sqrt(60000), 5), ")")
print("Spearman(RR, y*) :", rho(est, gt))

In [ ]:
%%writefile run_rr_holdout.py
"""RR-set baseline on the synthetic hold-out panel (same instances, same GT)."""
import json
import numpy as np
from scipy.stats import wilcoxon
from escape_lib import (make_instance, ground_truth, rho, timed,
                        alpha_from_rule, m_escape_pull, m_mc, m_irie)
from rr_ranking import m_rr

CAL = json.load(open("calibration.json"))
COEF = (CAL["intercept"], CAL["slope"])
TOPOS = ["er", "ba", "grid", "ws"]
RANGES = [(0.02, 0.08), (0.08, 0.18), (0.18, 0.38), (0.28, 0.52), (0.45, 0.72)]
SEEDS = [101, 102, 103, 104, 105]

rows = []
for topo in TOPOS:
    for (lo, hi) in RANGES:
        for sd in SEEDS:
            inst = make_instance(topo, 100, lo, hi, sd)
            gt = ground_truth(inst, 3000, seed=5000 + sd)
            rec = dict(topo=topo, lo=lo, hi=hi, seed=sd,
                       mean_pesc=inst.mean_pesc, rho={}, ms={})
            for nm, fn, kw in [
                ("rule-a", m_escape_pull, dict(alpha=alpha_from_rule(inst, COEF))),
                ("IRIE", m_irie, dict(alpha=0.7)),
                ("MC 200", m_mc, dict(n_samples=200)),
                ("RR 200", m_rr, dict(n_sets=200, seed=77 + sd)),
                ("RR 800", m_rr, dict(n_sets=800, seed=177 + sd)),
                ("RR 3200", m_rr, dict(n_sets=3200, seed=277 + sd)),
            ]:
                s, dt = timed(fn, inst, **kw)
                rec["rho"][nm] = rho(s, gt)
                rec["ms"][nm] = dt * 1000
            rows.append(rec)
    print("done", topo, flush=True)

json.dump(rows, open("rr_holdout.json", "w"), indent=1)
M = ["rule-a", "IRIE", "MC 200", "RR 200", "RR 800", "RR 3200"]
print(f"\n{'method':10s} {'mean rho':>9s} {'mean ms':>9s}")
for m in M:
    print(f"{m:10s} {np.mean([r['rho'][m] for r in rows]):9.4f} "
          f"{np.mean([r['ms'][m] for r in rows]):9.2f}")
mc = np.array([r["rho"]["MC 200"] for r in rows])
for m in ["RR 200", "RR 800", "RR 3200"]:
    v = np.array([r["rho"][m] for r in rows]); d = v - mc
    nz = d[np.abs(d) > 1e-12]
    print(f"{m} vs MC 200: {d.mean():+.4f}  wins {(d>0).sum()}/100  "
          f"p={wilcoxon(nz).pvalue:.2e}")


In [ ]:
import subprocess, sys, shutil, os
subprocess.run([sys.executable, "run_rr_holdout.py"], check=True)

In [ ]:
%%writefile real/run_rr_citation.py
"""RR-set baseline on the 35 real citation instances (same configs, same GT)."""
import json
import pickle
import sys
import numpy as np
from scipy.stats import wilcoxon

sys.path.insert(0, "..")
from escape_lib import ground_truth, rho, timed, m_mc
from rr_ranking import m_rr
from real_experiment import make_real_instance

topos = pickle.load(open("topologies.pkl", "rb"))
cfgs = json.load(open("real_configs.json"))
rows = []
for cfg in cfgs:
    ins = make_real_instance(topos, cfg)
    gt = ground_truth(ins, 3000, seed=cfg["seed"] + 7)
    rec = dict(cfg)
    rec.update(n=ins.n, m=int(ins.m), mean_pesc=float(ins.mean_pesc),
               rho={}, ms={})
    for nm, fn, kw in [("MC 50", m_mc, dict(n_samples=50)),
                       ("MC 200", m_mc, dict(n_samples=200)),
                       ("RR 200", m_rr, dict(n_sets=200, seed=cfg["seed"] + 31)),
                       ("RR 800", m_rr, dict(n_sets=800, seed=cfg["seed"] + 63))]:
        s, dt = timed(fn, ins, **kw)
        rec["rho"][nm] = rho(s, gt)
        rec["ms"][nm] = dt * 1000
    rows.append(rec)
    print(f"[{cfg['id']:2d}] {cfg['topo']:15s} Pesc={ins.mean_pesc:.3f} "
          f"MC200 {rec['rho']['MC 200']:.4f} ({rec['ms']['MC 200']:.0f}ms) "
          f"RR200 {rec['rho']['RR 200']:.4f} ({rec['ms']['RR 200']:.0f}ms) "
          f"RR800 {rec['rho']['RR 800']:.4f} ({rec['ms']['RR 800']:.0f}ms)",
          flush=True)

json.dump(rows, open("rr_citation.json", "w"), indent=1)
print(f"\n{'method':8s} {'mean rho':>9s} {'mean ms':>9s}")
for m in ["MC 50", "MC 200", "RR 200", "RR 800"]:
    print(f"{m:8s} {np.mean([r['rho'][m] for r in rows]):9.4f} "
          f"{np.mean([r['ms'][m] for r in rows]):9.1f}")
mc = np.array([r["rho"]["MC 200"] for r in rows])
for m in ["RR 200", "RR 800"]:
    d = np.array([r["rho"][m] for r in rows]) - mc
    print(f"{m} vs MC 200: {d.mean():+.4f}  wins {(d>0).sum()}/35  "
          f"p={wilcoxon(d[np.abs(d)>1e-12]).pvalue:.2e}")


In [ ]:
%%writefile real/run_rr_dense.py
"""RR-set baseline on the 30 dense real instances (same configs, same GT)."""
import json
import sys
import numpy as np
from scipy.stats import wilcoxon

sys.path.insert(0, "..")
from escape_lib import Instance, ground_truth, rho, timed, m_pesc, m_mc
from rr_ranking import m_rr
from scale_real import load_graph, bfs_ball, edges_of
from dense_panel import tune, TARGET, CFGS

import os
done = set()
if os.path.exists("rr_dense.jsonl"):
    done = {json.loads(l)["cfg_id"] for l in open("rr_dense.jsonl") if l.strip()}
out = open("rr_dense.jsonl", "a")
cache = {}
for c in CFGS:
    if c["id"] in done:
        continue
    if c["genre"] not in cache:
        cache.clear()
        cache[c["genre"]] = load_graph(c["genre"])
    G = cache[c["genre"]]
    if G.number_of_nodes() < c["n_t"]:
        continue
    S = bfs_ball(G, c["n_t"], seed=c["seed"])
    n = S.number_of_nodes()
    if n <= 100:
        continue                       # degenerate snowball samples, as before
    ei, ej = edges_of(S)
    rng = np.random.default_rng(c["seed"] + 5)
    x = rng.dirichlet(np.full(n, 0.7))
    x /= x.sum()
    p = tune(n, ei, ej, x, TARGET, c["seed"])
    inst = Instance(n, ei, ej, p, x)
    gt = ground_truth(inst, 3000, seed=c["seed"] * 7 + 1)
    rec = dict(cfg_id=c["id"], genre=c["genre"], n=n, m=int(inst.m),
               mean_pesc=float(inst.mean_pesc), rho={}, ms={})
    for nm, fn, kw in [("P_esc alone", m_pesc, {}),
                       ("MC 200", m_mc, dict(n_samples=200)),
                       ("RR 200", m_rr, dict(n_sets=200, seed=c["seed"] + 31)),
                       ("RR 800", m_rr, dict(n_sets=800, seed=c["seed"] + 63))]:
        s, dt = timed(fn, inst, **kw)
        rec["rho"][nm] = rho(s, gt)
        rec["ms"][nm] = dt * 1000
    out.write(json.dumps(rec) + "\n"); out.flush()
    print(f"{c['genre']:17s} n={n:6d} Pesc={inst.mean_pesc:.3f} | "
          f"Pesc {rec['rho']['P_esc alone']:.4f} "
          f"MC200 {rec['rho']['MC 200']:.4f} ({rec['ms']['MC 200']:.0f}ms) "
          f"RR200 {rec['rho']['RR 200']:.4f} ({rec['ms']['RR 200']:.0f}ms) "
          f"RR800 {rec['rho']['RR 800']:.4f}", flush=True)

out.close()
rows = [json.loads(l) for l in open("rr_dense.jsonl") if l.strip()]
json.dump(rows, open("rr_dense.json", "w"), indent=1)
print(f"\n{len(rows)} instances")
print(f"{'method':12s} {'mean rho':>9s} {'mean ms':>9s}")
for m in ["P_esc alone", "MC 200", "RR 200", "RR 800"]:
    print(f"{m:12s} {np.mean([r['rho'][m] for r in rows]):9.4f} "
          f"{np.mean([r['ms'][m] for r in rows]):9.1f}")
pe = np.array([r["rho"]["P_esc alone"] for r in rows])
mc = np.array([r["rho"]["MC 200"] for r in rows])
for m, base, bn in [("RR 200", mc, "MC 200"), ("RR 800", mc, "MC 200"),
                    ("RR 200", pe, "Pesc"), ("RR 800", pe, "Pesc")]:
    d = np.array([r["rho"][m] for r in rows]) - base
    nz = d[np.abs(d) > 1e-12]
    print(f"{m} vs {bn}: {d.mean():+.4f}  wins {(d>0).sum()}/{len(rows)}  "
          f"p={wilcoxon(nz).pvalue:.2e}")


In [ ]:
shutil.copy("rr_ranking.py", "real/rr_ranking.py")
os.chdir("real")
subprocess.run([sys.executable, "run_rr_citation.py"], check=True)
subprocess.run([sys.executable, "run_rr_dense.py"], check=True)
os.chdir("..")

In [ ]:
import json, numpy as np
from scipy.stats import wilcoxon

hold = json.load(open("rr_holdout.json"))
cit  = json.load(open("real/rr_citation.json"))
dens = json.load(open("real/rr_dense.json"))

print(f"{'':22s} {'synthetic (100)':>18s} {'citation (35)':>16s} {'dense real (30)':>17s}")
def cell(rows, m):
    if not rows or m not in rows[0]["rho"]: return f"{'---':>8s} {'':>7s}"
    return (f"{np.mean([r['rho'][m] for r in rows]):8.4f} "
            f"{np.mean([r['ms'][m] for r in rows]):6.0f}ms")
for m in ["MC 200", "RR 200", "RR 800", "RR 3200"]:
    print(f"{m:22s} {cell(hold, m)} {cell(cit, m)} {cell(dens, m)}")

for label, rows in [("synthetic", hold), ("citation", cit), ("dense", dens)]:
    mc = np.array([r["rho"]["MC 200"] for r in rows])
    for m in ["RR 200", "RR 800"]:
        d = np.array([r["rho"][m] for r in rows]) - mc
        nz = d[np.abs(d) > 1e-12]
        print(f"{label:10s} {m} vs MC 200: {d.mean():+.4f}  "
              f"wins {(d>0).sum()}/{len(d)}  p={wilcoxon(nz).pvalue:.2e}")

# the sparse-bucket collapse on citation networks
sparse = [r for r in cit if r["mean_pesc"] < 0.625]
print(f"\ncitation sparse bucket ({len(sparse)} inst): "
      f"RR-200 {np.mean([r['rho']['RR 200'] for r in sparse]):.4f} vs "
      f"MC-200 {np.mean([r['rho']['MC 200'] for r in sparse]):.4f}")

### What the RR baseline settles

**At matched budget, the possible-world estimator wins for whole-graph
ranking.** Synthetic hold-out: RR-200 trails MC-200 by $-0.133$ at the same
wall clock ($p=3.9\times10^{-16}$); $\sim$16$\times$ the sets are needed to
draw level. Citation networks: $-0.323$ ($p=2.9\times10^{-10}$), driven by a
sparse-bucket collapse (RR-200 $0.41$ vs MC-200 $0.97$).

**The cause is structural, not implementational.** One sampled world yields
component labels for *every* vertex — all $n$ estimates share it. One RR set
yields a single containment bit per vertex, and in sparse regimes most
vertices appear in no set at all and are left tied at zero. RR sets carry
$(\varepsilon,\delta)$ guarantees for $\sigma(S)$ of a **seed set** — the
task they were designed for — not for ranking all vertices.

**The dense regime inverts the picture, and deterministic percolation wins it
anyway.** At $\bar P_{esc}=0.90$ each set is nearly the giant component:
RR-800 beats MC-200 ($+0.015$, 30/30) at $8.5\times$ its cost — but does not
significantly beat the free $P_{esc}$ ($p=0.34$) at four orders of magnitude
more cost, and converged message passing (§14) outranks it outright at $1/59$
of its wall clock.